# ⚠️ Reanudación automática (Kaggle y Colab)

Todo lo que importa se guarda en almacenamiento persistente (`05_entrenamiento_v3/`):
- En **Kaggle**: Todo se guarda en `/kaggle/working/05_entrenamiento_v3/`.
- En **Colab**: Todo se guarda en `MiUnidad/entrenamiento_ia_ropa/05_entrenamiento_v3/`.

| Qué | Dónde | Cómo se reanuda |
|---|---|---|
| Análisis de cuerpos y prendas | `cache/` | cada archivo se escribe atómicamente; se rehace solo lo que faltó |
| Pares del profesor (CatVTON) | `pares/` + `filtro_registro.jsonl` | se saltan los aceptados **y** los rechazados |
| Entrenamiento | `checkpoints/ultimo_a.pth` / `ultimo_b.pth` | continúa en la época siguiente (modelo + optimizadores + calendario) |
| Mejor modelo | `checkpoints/mejor.pth` | no se sobrescribe si no mejora |

**Si la sesión se corta o reconecta:** Simplemente vuelve a ejecutar. Cada celda pesada detecta lo ya hecho y sigue por donde quedó.

**En Kaggle:** Recuerda activar en el panel lateral derecho:
- **Accelerator:** GPU T4 x2 (o GPU P100)
- **Internet:** **Internet on** (imprescindible para descargar librerías y pesos de CatVTON)


# FashionStore · Vestidor Virtual — Entrenamiento (Kaggle / Colab, < 2 h)

**CU32** · Web + móvil · Modo foto y modo cámara · GPU T4 / P100 gratuita

---

## Qué se entrena y qué no

Esto es **transfer learning por destilación**, no entrenamiento desde cero. Con 68 prendas y
sin un solo par real (una foto de *esta* persona con *esta* prenda puesta) no existe forma de
aprender try-on partiendo de nada; el intento anterior lo demuestra.

| Modo | Modelo | Entrenamiento | Latencia |
|---|---|---|---|
| **Foto** (la clienta sube su imagen) | CatVTON preentrenado (~13.000 pares VITON-HD) | ninguno, se usa tal cual | 8-15 s |
| **Cámara** (vídeo en vivo) | Red propia de ~1,9 M parámetros | **destilada del modo foto** | ~30 fps |

El modo foto genera la verdad de terreno que el modo cámara imita. Por eso basta una sola
sesión de entrenamiento.

---

## Presupuesto de tiempo (GPU T4 / P100)

| Fase | Tiempo |
|---|---|
| Dependencias | 6-8 min |
| Pose de los cuerpos | 2-3 min |
| Descarga de CatVTON | 6-8 min |
| **Destilación (420 pares)** | **40-45 min** |
| **Entrenamiento (40 épocas @ 192×256)** | **20-22 min** |
| Export ONNX verificado | 5 min |
| **Total** | **~1 h 25 min** |


---
## Celda 1 · Dependencias y GPU

In [ ]:
# Dependencias para el Python 3.13 / numpy 2.x que trae Colab hoy (comprobado con
# `pip install --dry-run` en Python 3.13). Los pines de 2024 (onnx 1.16.2, transformers
# 4.44, huggingface_hub 0.25) NO tienen rueda para 3.13 o chocan con CatVTON.
#
# MediaPipe NO se instala aqui: las poses vienen precalculadas de tu PC
# (06_poses_precalculadas/). Si esta instalado se usa de respaldo, pero no es necesario.
#
# Un fallo de instalacion se detiene con AssertionError (a diferencia de SystemExit, que
# NO frena "Ejecutar todo" y hacia reaparecer el error 20 min despues como ModuleNotFound).
import subprocess, sys, importlib, importlib.util, importlib.metadata, os, socket
from pathlib import Path

EN_KAGGLE = Path("/kaggle").is_dir()

def _hay_internet():
    try:
        socket.create_connection(("huggingface.co", 443), timeout=4)
        return True
    except Exception:
        return False

if EN_KAGGLE and not _hay_internet():
    raise RuntimeError(
        "⚠️ NO HAY ACCESO A INTERNET en este cuaderno de Kaggle.\n"
        "👉 En el panel lateral derecho de Kaggle:\n"
        "   1. Despliega 'Settings' (o 'Notebook options')\n"
        "   2. Busca 'Internet'\n"
        "   3. Cámbialo a 'Internet on'\n"
        "   (Si es tu primera vez, Kaggle pide verificar número telefónico)\n"
        "   Luego vuelve a ejecutar esta celda."
    )

, importlib.util, importlib.metadata, os

REQUISITOS = [
    ("diffusers",       "diffusers==0.31.0"),
    ("transformers",    "transformers==4.57.6"),
    ("huggingface_hub", "huggingface_hub>=0.34,<1.0"),
    ("accelerate",      "accelerate"),
    ("peft",            "peft>=0.13"),
    ("safetensors",     "safetensors"),
    ("cv2",             "opencv-python-headless"),
    ("onnx",            "onnx>=1.18"),
    ("onnxruntime",     "onnxruntime"),
]

def _version(mod):
    try:
        return importlib.metadata.version(mod)
    except Exception:
        return None

pendientes = []
for mod, pkg in REQUISITOS:
    if importlib.util.find_spec(mod) is None:
        pendientes.append(pkg)
# transformers debe ser exactamente la 4.57.x: la 5.x rompe el pipeline de CatVTON.
if importlib.util.find_spec("transformers") and not (_version("transformers") or "").startswith("4.57"):
    pendientes.append("transformers==4.57.6")
pendientes = sorted(set(pendientes))

if pendientes:
    print("Instalando:", ", ".join(pendientes))
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pendientes],
                       capture_output=True, text=True)
    assert r.returncode == 0, "INSTALACION FALLIDA:\n" + (r.stderr or r.stdout)[-1500:]
    print("Instalado. Si la siguiente comprobacion falla: Entorno de ejecucion -> "
          "Reiniciar sesion y vuelve a ejecutar desde la celda 1 (no pierdes nada: todo esta en Drive).")
else:
    print("Dependencias ya presentes.")

# Comprobacion real: que los modulos IMPORTEN, no que pip dijera que si.
importlib.invalidate_caches()
rotos = []
for mod, _ in REQUISITOS:
    try:
        importlib.import_module(mod)
    except Exception as e:
        rotos.append(f"{mod}: {e}")
try:
    from transformers import CLIPImageProcessor                       # lo importa CatVTON
    from diffusers import AutoencoderKL, UNet2DConditionModel, DDIMScheduler  # noqa: F401
except Exception as e:
    rotos.append(f"CatVTON (transformers/diffusers): {e}")
assert not rotos, ("No se pueden importar: " + " | ".join(rotos) +
                   "  ->  Entorno de ejecucion -> Reiniciar sesion y ejecuta de nuevo.")

# MediaPipe: OPCIONAL, nunca detiene nada.
try:
    import mediapipe as _mpipe
    print("mediapipe", _mpipe.__version__, "disponible (solo de respaldo).")
except Exception as e:
    print("mediapipe no disponible (no importa: se usan las poses precalculadas).")

import torch, numpy as np, cv2, json, csv, random, time, hashlib, unicodedata, shutil
import torch.nn as nn, torch.nn.functional as F
from pathlib import Path
from PIL import Image, ImageOps
from collections import defaultdict, Counter
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert DEV.type == "cuda", (
    "No hay GPU. Entorno de ejecucion -> Cambiar tipo de entorno -> Acelerador: GPU (T4)")
print(f"GPU   : {torch.cuda.get_device_name(0)}")
print(f"VRAM  : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print(f"torch : {torch.__version__}")

random.seed(42); np.random.seed(42); torch.manual_seed(42)


---
## Celda 2 · Entorno (Kaggle / Colab) y rutas

Estructura esperada:

```
01_cuerpos/                                  9 cuerpos HD + cuerpos_metadata.csv
02_prendas/<categoria>/                     68 prendas + prendas_manifest.json
04_checkpoints_modelos/
    resultados_vton_previos/cache/           1735 personas vestidas ya segmentadas
06_poses_precalculadas/                      poses de personas y prendas
dataset_manifest.json
```

- En **Kaggle**: se detecta automáticamente en `/kaggle/input/...` (solo lectura) y escribe en `/kaggle/working/`.
- En **Colab**: se detecta en Google Drive y escribe en `MiUnidad/entrenamiento_ia_ropa/`.


In [ ]:
import os, shutil
from pathlib import Path

EN_KAGGLE = Path("/kaggle").is_dir()
EN_COLAB = False

if not EN_KAGGLE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        EN_COLAB = True
    except Exception:
        EN_COLAB = False

RAIZ = None

if EN_KAGGLE:
    print("Detectado entorno Kaggle.")
    # Buscar recursivamente la carpeta que contiene 02_prendas en /kaggle/input
    for p in Path("/kaggle/input").rglob("02_prendas"):
        if p.is_dir():
            RAIZ = p.parent
            break
    if RAIZ is None:
        for p in Path("/kaggle/input").rglob("dataset_manifest.json"):
            if p.is_file():
                RAIZ = p.parent
                break
    if RAIZ is None:
        for cand in Path("/kaggle/input").glob("*"):
            if (cand / "02_prendas").is_dir() or (cand / "01_cuerpos").is_dir():
                RAIZ = cand
                break
            for sub in cand.glob("*"):
                if (sub / "02_prendas").is_dir() or (sub / "01_cuerpos").is_dir():
                    RAIZ = sub
                    break
            if RAIZ:
                break
else:
    CANDIDATAS = [
        Path("/content/drive/MyDrive/entrenamiento_ia_ropa"),
        Path("/content/drive/MyDrive/entrenamiento_ia_vestidor/entrenamiento_ia_ropa"),
        Path("/content/drive/My Drive/entrenamiento_ia_ropa"),
        Path("entrenamiento_ia_ropa"), Path("."),
    ]
    for c in CANDIDATAS:
        if (c / "02_prendas").is_dir() or (c / "01_cuerpos").is_dir():
            RAIZ = c
            break

assert RAIZ is not None, (
    "No encuentro la carpeta del dataset.\n"
    "- En Kaggle: agrega el dataset haciendo clic en '+ Add Input' (o sube el zip en Datasets).\n"
    "- En Colab: debe estar en MiUnidad/entrenamiento_ia_ropa/ con 01_cuerpos/ y 02_prendas/ dentro.")

DIR_CUERPOS  = RAIZ / "01_cuerpos"
DIR_PRENDAS  = RAIZ / "02_prendas"
DIR_VESTIDOS = RAIZ / "04_checkpoints_modelos" / "resultados_vton_previos" / "cache"

# En Kaggle, /kaggle/input es de SOLO LECTURA. Todo lo generado se guarda en /kaggle/working
if EN_KAGGLE:
    SALIDA = Path("/kaggle/working/05_entrenamiento_v3")
else:
    SALIDA = RAIZ / "05_entrenamiento_v3"

CACHE     = SALIDA / "cache"
DIR_PARES = SALIDA / "pares"
DIR_CKPT  = SALIDA / "checkpoints"
DIR_ONNX  = SALIDA / "onnx"
for p in (SALIDA, CACHE, DIR_PARES, DIR_CKPT, DIR_ONNX):
    p.mkdir(parents=True, exist_ok=True)

# Cache de personas vestidas
CACHE_VEST = Path("/content/cache_vestidos") if EN_COLAB else DIR_VESTIDOS
if EN_COLAB and DIR_VESTIDOS.is_dir() and not CACHE_VEST.exists():
    print("Copiando el cache de personas vestidas a disco local (mas rapido)...")
    shutil.copytree(DIR_VESTIDOS, CACHE_VEST)

# Poses precalculadas
DIR_POSES = RAIZ / "06_poses_precalculadas"
POSES = Path("/content/poses") if EN_COLAB else DIR_POSES
if EN_COLAB and DIR_POSES.is_dir() and not POSES.exists():
    shutil.copytree(DIR_POSES, POSES)

print(f"entorno  : {'Kaggle' if EN_KAGGLE else ('Colab' if EN_COLAB else 'Local')}")
print(f"raiz     : {RAIZ}")
print(f"poses    : {POSES.is_dir()} ({len(list(POSES.glob('*_pose.npy'))) if POSES.is_dir() else 0} archivos)")
print(f"cuerpos  : {DIR_CUERPOS.is_dir()}")
print(f"prendas  : {DIR_PRENDAS.is_dir()}")
print(f"vestidos : {CACHE_VEST.is_dir()}")
print(f"salida   : {SALIDA}")


In [ ]:
# ===== PANEL DE PROGRESO: que esta hecho y que toca ahora ================================
def panel_progreso():
    n_pares = len(list(DIR_PARES.glob("*_res.png")))
    n_vest = len(list(DIR_PARES.glob("vest_*_res.png")))
    ultimos = [p for p in (DIR_CKPT / "ultimo_a.pth", DIR_CKPT / "ultimo_b.pth") if p.exists()]
    ep = None
    if ultimos:
        try:
            ep = max(torch.load(p, map_location="cpu", weights_only=False)["epoca"] for p in ultimos)
        except Exception:
            ep = "?"
    pasos = [
        ("Poses precalculadas en Drive", POSES.is_dir() and len(list(POSES.glob("*_pose.npy"))) > 500),
        ("Pares aceptados (>=150, >=80 vestidas)", n_pares >= 150 and n_vest >= 80),
        ("Entrenamiento completo", (DIR_CKPT / "COMPLETO.json").exists()),
        ("Puerta B aprobada", (SALIDA / "puerta_b.json").exists() and
            json.loads((SALIDA / "puerta_b.json").read_text()).get("aprobada", False)),
        ("ONNX exportado", (DIR_ONNX / "vestidor_camara.onnx").exists()),
    ]
    print(f"Pares: {n_pares} (vestidas {n_vest})  ·  ultima epoca guardada: {ep if ep else 'ninguna'}")
    for nombre, ok in pasos:
        print(f"  [{'x' if ok else ' '}] {nombre}")
    sig = next((n for n, ok in pasos if not ok), None)
    print("\nSIGUIENTE:", sig or "todo listo")
panel_progreso()

---
## Celda 3 · Catálogo, cuerpos y personas vestidas

Tres fuentes:

- **68 prendas** del catálogo, con su `neck_type` y `sleeve_length`.
- **9 cuerpos HD** (768×1376) del catálogo, en bodysuit.
- **1735 personas vestidas** (192×256) con segmentación ya calculada — las que hacen que el
  modelo funcione con clientas reales.

Se filtra contra el manifiesto: cualquier `.png` que no sea una prenda (gráficas, vistas
previas) queda fuera. Sin ese filtro se llegaba a destilar una figura de matplotlib.

In [ ]:
def norm(s):
    s = unicodedata.normalize("NFD", str(s or ""))
    return "".join(c for c in s if unicodedata.category(c) != "Mn").lower().strip()

EXT = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

# --- manifiesto de prendas ---------------------------------------------------------
f_man = None
for cand in [DIR_PRENDAS / "prendas_manifest.json", RAIZ / "dataset_manifest.json"]:
    if cand.exists():
        f_man = cand
        break
assert f_man, "No encuentro prendas_manifest.json ni dataset_manifest.json"
manifest = json.loads(f_man.read_text(encoding="utf-8"))
POR_LIMPIO = {(v.get("clean_name") or k): dict(v) for k, v in manifest.items()}
print(f"Manifiesto: {f_man.name}  ({len(manifest)} prendas)")

# --- prendas -----------------------------------------------------------------------
PRENDAS, descartadas = [], []
for sub in sorted(DIR_PRENDAS.iterdir()):
    if not sub.is_dir():
        continue
    for img in sorted(sub.iterdir()):
        if img.suffix.lower() not in EXT or img.name.endswith("_parse.png"):
            continue
        if img.name not in POR_LIMPIO:          # solo prendas del manifiesto
            descartadas.append(img.name)
            continue
        m = dict(POR_LIMPIO[img.name])
        m.setdefault("category", sub.name)
        m.setdefault("neck_type", "N/A")
        m.setdefault("sleeve_length", "N/A")
        m.setdefault("parent_category", "Ropa Superior")
        m["archivo"] = img.name
        PRENDAS.append((img, m))

if descartadas:
    print(f"Descartados (fuera del manifiesto): {descartadas}")

def es_torso(m):
    return "inferior" not in norm(m.get("parent_category", ""))

PRENDAS_TORSO = [(r, m) for r, m in PRENDAS if es_torso(m)]
print(f"Prendas: {len(PRENDAS)}  ·  de torso: {len(PRENDAS_TORSO)}")
for c, n in sorted(Counter(m["category"] for _, m in PRENDAS_TORSO).items()):
    print(f"   {c:30s} {n}")

# --- cuerpos HD del catalogo --------------------------------------------------------
CUERPOS_HD = []
f_csv = DIR_CUERPOS / "cuerpos_metadata.csv"
if f_csv.exists():
    with open(f_csv, encoding="utf-8-sig") as f:
        for row in csv.DictReader(f):
            p = DIR_CUERPOS / row.get("filename", "")
            if p.exists():
                CUERPOS_HD.append((p, row))
for img in sorted(DIR_CUERPOS.iterdir()):
    if img.suffix.lower() in EXT and not any(img == c[0] for c in CUERPOS_HD):
        CUERPOS_HD.append((img, {"name": img.stem}))
print(f"\nCuerpos HD del catalogo: {len(CUERPOS_HD)}")

# --- personas vestidas (clave para que funcione con clientas reales) ----------------
VESTIDOS = []
if CACHE_VEST.is_dir():
    jpgs = {p.stem for p in CACHE_VEST.glob("*.jpg")}
    pars = {p.name[:-10] for p in CACHE_VEST.glob("*_parse.png")}
    VESTIDOS = sorted(jpgs & pars)      # solo los que tienen imagen Y segmentacion
print(f"Personas vestidas utilizables: {len(VESTIDOS)}")
assert len(VESTIDOS) > 200, (
    "Faltan las personas vestidas. Sube tambien "
    "04_checkpoints_modelos/resultados_vton_previos/cache/ a Drive: sin ellas el modo camara "
    "solo sabra vestir maniquies en bodysuit.")

---
## Celda 4 · Segmentación y pose

Dos modelos congelados. **SegFormer B2 Clothes** da las 18 clases (cara, pelo, ropa superior,
brazos, piernas). **MediaPipe Pose** da 33 puntos del esqueleto — esto es lo nuevo: el modelo
anterior no usaba pose en ningún momento, y por eso su única referencia geométrica era la ropa
que la persona ya llevaba puesta. De ahí nacía el bug principal.

Dos resoluciones:

- **384×512** para CatVTON (modo foto, calidad).
- **192×256** para el modelo de cámara. Cuatro veces menos píxeles que 384×512, que es lo que
  permite pasar de ~12 fps a más de 30 en el móvil.

Las 1735 personas vestidas ya traen su segmentación del entrenamiento anterior; solo hay que
calcularles la pose.

In [ ]:
from transformers import SegformerImageProcessor, SegformerForSemanticSegmentation

W_DIF, H_DIF = 384, 512      # CatVTON  (modo foto)
W, H         = 192, 256      # modelo ligero (modo camara)

L = {"background":0,"hat":1,"hair":2,"sunglasses":3,"upper-clothes":4,"skirt":5,"pants":6,
     "dress":7,"belt":8,"left-shoe":9,"right-shoe":10,"face":11,"left-leg":12,"right-leg":13,
     "left-arm":14,"right-arm":15,"bag":16,"scarf":17}

P = dict(nariz=0, boca_i=9, boca_d=10, hombro_i=11, hombro_d=12, codo_i=13, codo_d=14,
         muneca_i=15, muneca_d=16, cadera_i=23, cadera_d=24, rodilla_i=25, rodilla_d=26,
         tobillo_i=27, tobillo_d=28)

_seg = {"proc": None, "net": None}
def _cargar_seg():
    if _seg["net"] is None:
        mid = "mattmdjaga/segformer_b2_clothes"
        _seg["proc"] = SegformerImageProcessor.from_pretrained(mid)
        _seg["net"] = SegformerForSemanticSegmentation.from_pretrained(mid).to(DEV).eval()
    return _seg["proc"], _seg["net"]

@torch.no_grad()
def segmentar(img):
    proc, net = _cargar_seg()
    ent = proc(images=img, return_tensors="pt").to(DEV)
    lg = net(**ent).logits
    lg = F.interpolate(lg, size=(img.height, img.width), mode="bilinear", align_corners=False)
    return lg.argmax(1)[0].cpu().numpy().astype(np.uint8)

# --- POSE -------------------------------------------------------------------------------
# Orden, del mas al menos fiable:
#   1) PRECALCULADA en tu PC con MediaPipe (06_poses_precalculadas/): no depende de que
#      MediaPipe funcione en Colab. Es la via normal.
#   2) MediaPipe en Colab, SOLO CPU y con el .task local de 00_modelos_base (sin descargar).
#   3) Estimador por segmentacion. RESPALDO: ~20% de error en hombros. Se cuenta en
#      ORIGEN_POSE y la celda de control detiene todo si pasa del 15%.
try:
    import mediapipe as mpipe
    HAY_MP = True
except Exception as _e:
    mpipe, HAY_MP = None, False
    print("AVISO: mediapipe no esta disponible (" + str(_e)[:60] + "). No es un problema si "
          "las poses precalculadas estan en Drive.")

ORIGEN_POSE = Counter()
_pose = {"modo": None, "det": None}
MODELO_POSE = RAIZ / "00_modelos_base" / "pose_landmarker_lite.task"


def _iniciar_pose():
    """Elige backend: solutions (mediapipe 0.10.x), tasks (1.x, CPU, modelo local) o parse."""
    if not HAY_MP:
        _pose["modo"] = "parse"
    elif hasattr(mpipe, "solutions") and hasattr(mpipe.solutions, "pose"):
        _pose["modo"] = "solutions"
        _pose["det"] = mpipe.solutions.pose.Pose(
            static_image_mode=True, model_complexity=1,
            enable_segmentation=False, min_detection_confidence=0.3)
    else:
        try:
            from mediapipe.tasks import python as mp_py
            from mediapipe.tasks.python import vision
            assert MODELO_POSE.is_file(), f"falta {MODELO_POSE}"
            _pose["det"] = vision.PoseLandmarker.create_from_options(
                vision.PoseLandmarkerOptions(
                    base_options=mp_py.BaseOptions(
                        model_asset_path=str(MODELO_POSE),
                        delegate=mp_py.BaseOptions.Delegate.CPU),   # Colab no tiene GPU-GL
                    running_mode=vision.RunningMode.IMAGE, num_poses=1,
                    min_pose_detection_confidence=0.3))
            _pose["modo"] = "tasks"
        except Exception as e:
            print("No se pudo iniciar PoseLandmarker:", e)
            _pose["modo"] = "parse"
    print("Pose (respaldo) ->", {"solutions": "MediaPipe solutions", "tasks": "MediaPipe tasks CPU",
                                 "parse": "segmentacion (RESPALDO, menos preciso)"}[_pose["modo"]])
    return _pose["modo"]


def pose_precalculada(ident, ancho, alto, caja=None):
    """Lee la pose de 06_poses_precalculadas/<ident>_pose.npy (x, y normalizados) y la
    lleva a pixeles de la imagen de trabajo. `caja` = (x0, y0, ancho, alto) del contenido
    dentro de un letterbox."""
    f = POSES / f"{ident}_pose.npy"
    if not f.exists():
        return None
    p = np.load(f).astype(np.float32)
    if caja is None:
        p[:, 0] *= ancho; p[:, 1] *= alto
    else:
        ox, oy, cw, ch = caja
        p[:, 0] = ox + p[:, 0] * cw; p[:, 1] = oy + p[:, 1] * ch
    return p


def pose_de(img, parse=None, ident=None, caja=None):
    """33 landmarks [x, y, visibilidad] en pixeles de `img`, o None."""
    if ident is not None:
        p = pose_precalculada(ident, img.width, img.height, caja)
        if p is not None:
            ORIGEN_POSE["precalculada"] += 1
            return p
    if _pose["modo"] is None:
        _iniciar_pose()
    if _pose["modo"] != "parse":
        arr = np.asarray(img.convert("RGB"))
        h, w = arr.shape[:2]
        if _pose["modo"] == "solutions":
            r = _pose["det"].process(arr)
            lm = r.pose_landmarks.landmark if r.pose_landmarks else None
        else:
            r = _pose["det"].detect(mpipe.Image(image_format=mpipe.ImageFormat.SRGB, data=arr))
            lm = r.pose_landmarks[0] if r.pose_landmarks else None
        if lm is not None:
            ORIGEN_POSE["mediapipe"] += 1
            return np.array([[q.x * w, q.y * h, getattr(q, "visibility", 1.0)] for q in lm],
                            dtype=np.float32)
    if parse is None:
        return None
    p = pose_de_parse(parse)
    if p is not None:
        ORIGEN_POSE["respaldo"] += 1
    return p


def guardar_png(img, ruta):
    """Escritura atomica: si Colab se corta a mitad no queda un PNG corrupto."""
    tmp = ruta.with_name(ruta.name + ".tmp")
    img.save(tmp, format="PNG")
    tmp.replace(ruta)

def guardar_npy(arr, ruta):
    tmp = ruta.with_name(ruta.stem + ".tmp.npy")
    np.save(tmp, arr)
    tmp.replace(ruta)


def _mayor(m):
    m = m.astype(np.uint8)
    n, et, st, _ = cv2.connectedComponentsWithStats(m, 8)
    if n <= 1:
        return m
    return (et == 1 + int(np.argmax(st[1:, cv2.CC_STAT_AREA]))).astype(np.uint8)

def _cadena_brazo(mb, hombro):
    """3 puntos (hombro, codo, muneca) siguiendo el eje principal del brazo."""
    ys, xs = np.nonzero(mb)
    if ys.size < 25:
        return None
    pts = np.stack([xs, ys], 1).astype(np.float32)
    c = pts.mean(0)
    _, _, vt = np.linalg.svd(pts - c, full_matrices=False)
    ax = vt[0]
    t = (pts - c) @ ax
    # orientar: t creciente = alejandose del hombro
    if np.linalg.norm((c + ax * t.min()) - hombro) > np.linalg.norm((c + ax * t.max()) - hombro):
        ax, t = -ax, -t
    q = np.quantile(t, [0.0, 0.18, 0.50, 0.82, 1.0])
    seg = []
    for a, b in [(q[0], q[1]), (q[1], q[3]), (q[3], q[4])]:
        sel = (t >= a) & (t <= b)
        seg.append(pts[sel].mean(0) if sel.sum() >= 3 else c + ax * ((a + b) / 2))
    return seg  # [cerca del hombro, codo, muneca]

def pose_de_parse(parse):
    """Devuelve 33 landmarks [x, y, vis] compatibles con el indice de MediaPipe.

    Solo rellena los que usa el cuaderno; el resto queda a cero con visibilidad 0.
    """
    h, w = parse.shape
    lm = np.zeros((33, 3), np.float32)

    torso = _mayor(np.isin(parse, [L["upper-clothes"], L["dress"]]))
    if torso.sum() < 200:
        torso = _mayor(np.isin(parse, [L["upper-clothes"], L["dress"], L["scarf"]]))
    if torso.sum() < 200:
        return None

    ys, xs = np.nonzero(torso)
    y0, y1 = int(ys.min()), int(ys.max())
    anchos = torso.sum(1)

    # --- hombros: primera fila del tercio superior que alcanza el 88% del ancho maximo ---
    lim = y0 + max(2, int(0.35 * (y1 - y0)))
    tramo = anchos[y0:lim + 1]
    obj = 0.88 * float(tramo.max()) if tramo.size else 0.0
    fila = y0 + int(np.argmax(tramo >= obj)) if tramo.size else y0
    col = np.nonzero(torso[fila])[0]
    if col.size < 4:
        fila = y0 + (lim - y0) // 2
        col = np.nonzero(torso[fila])[0]
    xi, xd = float(col.min()), float(col.max())
    ancho_h = xd - xi
    # Los hombros anatomicos estan algo por dentro del borde de la prenda.
    # OJO al lado: "izquierdo" es el anatomico, que en la imagen cae a la DERECHA.
    # MediaPipe y las etiquetas de SegFormer usan esa misma convencion; invertirla
    # sembraria la cadena del brazo desde el hombro opuesto.
    hom_d = np.array([xi + 0.08 * ancho_h, float(fila)], np.float32)
    hom_i = np.array([xd - 0.08 * ancho_h, float(fila)], np.float32)
    esc = float(np.linalg.norm(hom_i - hom_d)) or (w * 0.25)

    # --- caderas ---
    inf = _mayor(np.isin(parse, [L["pants"], L["skirt"]]))
    if inf.sum() > 150:
        yi = np.nonzero(inf)[0]
        fila_c = int(yi.min() + 0.06 * (yi.max() - yi.min()))
    else:
        fila_c = min(h - 1, int(fila + 1.55 * esc))
    fila_c = min(h - 1, max(fila + 3, fila_c))
    base = inf if inf.sum() > 150 else torso
    colc = np.nonzero(base[min(fila_c, h - 1)])[0]
    if colc.size >= 4:
        cad_d = np.array([float(colc.min()) + 0.06 * esc, float(fila_c)], np.float32)
        cad_i = np.array([float(colc.max()) - 0.06 * esc, float(fila_c)], np.float32)
    else:
        cad_d = np.array([min(hom_i[0], hom_d[0]) + 0.05 * esc, float(fila_c)], np.float32)
        cad_i = np.array([max(hom_i[0], hom_d[0]) - 0.05 * esc, float(fila_c)], np.float32)

    # --- boca: parte baja de la cara ---
    cara = np.isin(parse, [L["face"]])
    if cara.sum() > 60:
        yc, xc = np.nonzero(cara)
        y_boca = float(np.quantile(yc, 0.86))
        x_boca = float(xc[yc >= np.quantile(yc, 0.72)].mean())
    else:
        cab = np.isin(parse, [L["hair"], L["hat"], L["face"], L["sunglasses"]])
        if cab.sum() > 60:
            yc, xc = np.nonzero(cab)
            y_boca, x_boca = float(np.quantile(yc, 0.92)), float(xc.mean())
        else:
            y_boca, x_boca = float(fila) - 0.42 * esc, float((hom_i[0] + hom_d[0]) / 2)
    y_boca = min(y_boca, float(fila) - 0.12 * esc)

    # --- tobillos ---
    pies = np.isin(parse, [L["left-shoe"], L["right-shoe"]])
    piernas = np.isin(parse, [L["left-leg"], L["right-leg"]])
    if pies.sum() > 80:
        yp, xp = np.nonzero(pies)
        y_tob = float(np.quantile(yp, 0.25)); x_tob = float(xp.mean())
    elif piernas.sum() > 200:
        yp, xp = np.nonzero(piernas)
        y_tob = float(yp.max()); x_tob = float(xp.mean())
    else:
        y_tob = float(fila) + 3.6 * esc; x_tob = float((cad_i[0] + cad_d[0]) / 2)

    # --- brazos ---
    br = {}
    for lado, etiqueta, hom in (("i", "left-arm", hom_i), ("d", "right-arm", hom_d)):
        m = _mayor(np.isin(parse, [L[etiqueta]]))
        cad = _cadena_brazo(m, hom) if m.sum() > 25 else None
        if cad is None:
            # sin brazo visible: se supone colgando en vertical
            fuera = 1.0 if lado == "i" else -1.0
            cad = [hom, hom + np.array([fuera * 0.06 * esc, 0.78 * esc], np.float32),
                   hom + np.array([fuera * 0.10 * esc, 1.52 * esc], np.float32)]
        br[lado] = cad

    def poner(k, p, vis=1.0):
        lm[P[k]] = [float(p[0]), float(p[1]), vis]

    poner("hombro_i", hom_i); poner("hombro_d", hom_d)
    poner("cadera_i", cad_i); poner("cadera_d", cad_d)
    poner("boca_i", [x_boca - 0.06 * esc, y_boca]); poner("boca_d", [x_boca + 0.06 * esc, y_boca])
    poner("nariz", [x_boca, y_boca - 0.10 * esc])
    poner("tobillo_d", [x_tob - 0.10 * esc, y_tob]); poner("tobillo_i", [x_tob + 0.10 * esc, y_tob])
    poner("rodilla_i", [(cad_i[0] + x_tob) / 2, (cad_i[1] + y_tob) / 2])
    poner("rodilla_d", [(cad_d[0] + x_tob) / 2, (cad_d[1] + y_tob) / 2])
    for lado in ("i", "d"):
        poner(f"codo_{lado}", br[lado][1]); poner(f"muneca_{lado}", br[lado][2])
    return lm


def letterbox(im, w=W_DIF, h=H_DIF, relleno=(255, 255, 255)):
    im = im.convert("RGB")
    e = min(w / im.width, h / im.height)
    nv = im.resize((max(1, int(im.width * e)), max(1, int(im.height * e))), Image.LANCZOS)
    li = Image.new("RGB", (w, h), relleno)
    li.paste(nv, ((w - nv.width) // 2, (h - nv.height) // 2))
    return li

def _caja_letterbox(ow, oh, w=W_DIF, h=H_DIF):
    """Donde cae el contenido de una imagen ow x oh dentro de su letterbox (igual que arriba)."""
    e = min(w / ow, h / oh)
    nw, nh = max(1, int(ow * e)), max(1, int(oh * e))
    return ((w - nw) // 2, (h - nh) // 2, nw, nh)

def clave(p):
    return hashlib.md5(str(Path(p).name).encode()).hexdigest()[:16]

def ident_hd(ruta):
    """Identificador de un cuerpo HD en 06_poses_precalculadas."""
    return f"hd_{Path(ruta).stem}"

def analizar(ruta, ident=None):
    """Cuerpos del catalogo y prendas: segmenta (+ pose si se da `ident`), cacheado en Drive.

    El marcador de "terminado" es la imagen, que se escribe la ULTIMA y de forma atomica:
    si Colab se corta, la siguiente ejecucion rehace solo lo que faltaba.
    """
    k = clave(ruta)
    fi, fp, fq = CACHE / f"{k}.png", CACHE / f"{k}_parse.png", CACHE / f"{k}_pose.npy"
    if fi.exists() and fp.exists():
        img = Image.open(fi).convert("RGB")
        par = np.array(Image.open(fp), dtype=np.uint8)
        pos = np.load(fq) if fq.exists() else None
        if pos is None and ident is not None:
            ow, oh = ImageOps.exif_transpose(Image.open(ruta)).size
            pos = pose_de(img, par, ident, _caja_letterbox(ow, oh))
            if pos is not None:
                guardar_npy(pos, fq)
        return img, par, pos
    orig = ImageOps.exif_transpose(Image.open(ruta))
    img = letterbox(orig)
    par = segmentar(img)
    pos = pose_de(img, par, ident, _caja_letterbox(*orig.size)) if ident is not None else None
    guardar_png(Image.fromarray(par), fp)
    if pos is not None:
        guardar_npy(pos, fq)
    guardar_png(img, fi)
    return img, par, pos

def analizar_vestida(k):
    """Personas vestidas del cache previo: ya traen parse; la pose viene precalculada.

    Se amplian a 384x512 porque MediaPipe pierde fiabilidad a 192x256.
    """
    fq = CACHE / f"v_{k}_pose.npy"
    img = Image.open(CACHE_VEST / f"{k}.jpg").convert("RGB").resize((W_DIF, H_DIF), Image.LANCZOS)
    par = np.array(Image.open(CACHE_VEST / f"{k}_parse.png").resize((W_DIF, H_DIF), Image.NEAREST),
                   dtype=np.uint8)
    if fq.exists():
        return img, par, np.load(fq)
    pos = pose_de(img, par, ident=f"v_{k}")
    if pos is not None:
        guardar_npy(pos, fq)
    return img, par, pos

print("Analizando prendas del catalogo...")
for p, _ in tqdm(PRENDAS_TORSO):
    analizar(p)
print("Analizando cuerpos HD...")
for p, _ in tqdm(CUERPOS_HD):
    analizar(p, ident_hd(p))

sin_pose = [m.get("name", p.stem) for p, m in CUERPOS_HD if analizar(p, ident_hd(p))[2] is None]
print(f"\nCuerpos HD sin pose: {sin_pose if sin_pose else 'ninguno'}")


---
# Celda 5 · ⭐ La corrección: máscara agnóstica condicionada a la prenda destino

Aquí está la reparación. Todo lo demás se apoya en esta celda.

> La región que se borra del cuerpo debe ser un **superconjunto** de la silueta que la prenda
> nueva podría ocupar. Si se borra menos, la prenda se recorta. Nunca debe depender de la
> prenda que la persona ya lleva puesta.

Tres fuentes, ninguna de ellas la ropa vieja como plantilla:

1. **Pose** → hombros, codos, muñecas, caderas, barbilla. El esqueleto del torso y los brazos,
   independiente de lo que la persona vista.
2. **`neck_type` de la prenda destino** → cuánto se libera hacia la barbilla.
   `Cuello Alto con Cierre 1/4` libera hasta la barbilla; `Escote en V Profundo` casi nada.
3. **`sleeve_length` de la prenda destino** → qué fracción del brazo se libera.
   `Manga Larga` hasta la muñeca; `Sin Mangas` solo el hombro.

```
   barbilla ·······           cuello=1.00  cuello alto / tortuga
            ·······           cuello=0.85  camisero / solapa / polo
   hombros  ███████           cuello=0.70  redondo / barco / cuadrado
            ███████           cuello=0.35  escote en V
            ███████           cuello=0.15  V profundo
   codos    ██ manga=0.45     corta / cap
   muñecas  █  manga=1.00     larga
   caderas  ███████           largo=0.42   ropa superior
   rodillas ███████           largo=0.78   vestido / enterizo
```

**Para la clienta que se fotografía con ropa normal**, la ropa que lleva se **une** a esa zona
(dilatada), no la define: así una sudadera ancha desaparece entera aunque la prenda destino sea
un top de tirantes, y el hueco de piel que queda lo rellena el generador. Ese cambio de rol —de
plantilla a garantía de borrado— es exactamente lo que faltaba.

In [ ]:
def plan_de_borrado(meta):
    """Metadatos de la prenda -> tres fracciones geometricas.

    cuello : 0 = no subir del hombro, 1 = liberar hasta la barbilla
    manga  : fraccion del brazo (hombro->muneca) que se libera
    largo  : fraccion del cuerpo (hombro->tobillo) que se libera hacia abajo
    """
    n, s = norm(meta.get("neck_type")), norm(meta.get("sleeve_length"))
    cat, padre = norm(meta.get("category")), norm(meta.get("parent_category"))

    if any(k in n for k in ("alto", "tortuga", "cierre 1/4", "cerrado")):
        cuello = 1.00
    elif any(k in n for k in ("camisero", "solapa", "polo")):
        cuello = 0.85
    elif "henley" in n:                      # redondo con botonadura, no camisero
        cuello = 0.70
    elif "profundo" in n:
        cuello = 0.15
    elif "en v" in n or n.startswith("v "):
        cuello = 0.35
    elif any(k in n for k in ("corazon", "tirantes", "asimetrico", "un hombro", "peto")):
        cuello = 0.45
    elif any(k in n for k in ("redondo", "barco", "recto", "cuadrado")):
        cuello = 0.70
    else:
        cuello = 0.70                        # N/A -> valor amplio, que es el seguro

    if "larga" in s:
        manga = 1.00
    elif "sin mangas" in s and "cap" not in s:
        manga = 0.14
    else:
        manga = 0.45

    # Chaquetas y cardigans van encima de todo: liberan mas que su escote nominal.
    if "chaqueta" in cat or "exterior" in padre or "cardigan" in norm(meta.get("archivo", "")):
        cuello, manga = max(cuello, 0.80), max(manga, 0.85)

    if "una pieza" in padre or any(k in cat for k in ("vestido", "enterizo", "mono")):
        largo = 0.78
    elif "inferior" in padre or any(k in cat for k in ("jean", "pantalon", "falda")):
        largo = 1.00
    elif "descanso" in padre or "pijama" in cat:
        largo = 0.78
    else:
        largo = 0.42

    solo_inferior = ("inferior" in padre) or any(k in cat for k in ("jean", "pantalon", "falda"))
    return dict(cuello=cuello, manga=manga, largo=largo, solo_inferior=solo_inferior)


def _capsula(lienzo, a, b, radio):
    """Segmento grueso de extremos redondeados: un brazo."""
    a, b = tuple(np.round(a[:2]).astype(int)), tuple(np.round(b[:2]).astype(int))
    cv2.line(lienzo, a, b, 1, thickness=int(max(2, radio * 2)), lineType=cv2.LINE_AA)
    cv2.circle(lienzo, a, int(radio), 1, -1, lineType=cv2.LINE_AA)
    cv2.circle(lienzo, b, int(radio), 1, -1, lineType=cv2.LINE_AA)


def mascara_agnostica_por_prenda(parse, pose, meta, margen=1.0):
    """Zona a regenerar. Depende de la POSE y de la PRENDA DESTINO, nunca de la ropa actual."""
    h, w = parse.shape
    plan = plan_de_borrado(meta)
    zona = np.zeros((h, w), np.uint8)
    ancho = w * 0.25

    cara_pelo = np.isin(parse, [L["hat"], L["hair"], L["sunglasses"], L["face"]])
    ropa_sup  = np.isin(parse, [L["upper-clothes"], L["dress"]])
    ropa_inf  = np.isin(parse, [L["skirt"], L["pants"], L["belt"]])

    if pose is not None:
        pt = lambda k: pose[P[k]][:2]
        hi, hd = pt("hombro_i"), pt("hombro_d")
        ci, cd = pt("cadera_i"), pt("cadera_d")
        ancho = float(np.linalg.norm(hi - hd)) or (w * 0.25)

        boca = (pt("boca_i") + pt("boca_d")) / 2.0
        barbilla = boca + np.array([0.0, 0.10 * ancho])
        y_hombro = (hi[1] + hd[1]) / 2.0
        y_tope = y_hombro + plan["cuello"] * (barbilla[1] - y_hombro)

        tob = (pt("tobillo_i") + pt("tobillo_d")) / 2.0
        y_fondo = min(y_hombro + plan["largo"] * (tob[1] - y_hombro), h - 1.0)

        ex_h = 0.20 * ancho * margen     # hombros mas anchos: prendas oversize
        ex_c = 0.18 * ancho * margen     # caderas mas anchas: vuelo, peplum
        izq, der = (hi, hd) if hi[0] < hd[0] else (hd, hi)
        c_izq, c_der = (ci, cd) if ci[0] < cd[0] else (cd, ci)

        torso = np.array([
            [izq[0] - ex_h, y_tope], [der[0] + ex_h, y_tope],
            [der[0] + ex_h, (der[1] + c_der[1]) / 2.0], [c_der[0] + ex_c, y_fondo],
            [c_izq[0] - ex_c, y_fondo], [izq[0] - ex_h, (izq[1] + c_izq[1]) / 2.0],
        ], np.int32)
        cv2.fillPoly(zona, [torso], 1, lineType=cv2.LINE_AA)

        if not plan["solo_inferior"]:
            for lado in ("i", "d"):
                ho, co, mu = pt(f"hombro_{lado}"), pt(f"codo_{lado}"), pt(f"muneca_{lado}")
                l1, l2 = np.linalg.norm(co - ho), np.linalg.norm(mu - co)
                alcance = plan["manga"] * (l1 + l2 + 1e-6)
                radio = 0.17 * ancho * margen
                if alcance <= l1:
                    _capsula(zona, ho, ho + (co - ho) * (alcance / (l1 + 1e-6)), radio)
                else:
                    _capsula(zona, ho, co, radio)
                    _capsula(zona, co, co + (mu - co) * ((alcance - l1) / (l2 + 1e-6)), radio * 0.80)
    else:
        ys, xs = np.nonzero(ropa_sup)
        if ys.size:
            y0, y1, x0, x1 = ys.min(), ys.max(), xs.min(), xs.max()
            ah, aw = y1 - y0, x1 - x0
            y0 = max(0, int(y0 - plan["cuello"] * 0.30 * ah))
            y1 = min(h - 1, int(y0 + plan["largo"] / 0.42 * ah))
            x0, x1 = max(0, int(x0 - 0.35 * aw)), min(w - 1, int(x1 + 0.35 * aw))
            zona[y0:y1 + 1, x0:x1 + 1] = 1

    # Redondeo: el polígono de torso + capsulas de brazo dejaba hombros con esquinas rectas.
    # Ese recuadro era lo que luego se veia como una "caja" en el resultado. Se suaviza ANTES
    # de la garantia de borrado, y con umbral 0.35 (algo generoso) para no destapar ropa previa.
    sigma = max(2.0, 0.06 * (ancho if pose is not None else w * 0.25))
    zona = (cv2.GaussianBlur(zona.astype(np.float32), (0, 0), sigma) > 0.35).astype(np.uint8)

    # GARANTIA de borrado: la ropa que la clienta lleva puesta desaparece entera aunque
    # sobresalga del recuadro (sudadera ancha, chaqueta). Union, NO plantilla: esta es la
    # diferencia con el modelo roto.
    previa = ropa_sup if not plan["solo_inferior"] else ropa_inf
    previa = cv2.dilate(previa.astype(np.uint8), np.ones((11, 11), np.uint8), 1)
    zona = np.clip(zona + previa, 0, 1).astype(np.uint8)

    zona[cv2.dilate(cara_pelo.astype(np.uint8), np.ones((3, 3), np.uint8), 1) > 0] = 0

    if plan["largo"] < 0.60 and not plan["solo_inferior"]:
        conservar = np.isin(parse, [L["skirt"], L["pants"], L["left-leg"], L["right-leg"],
                                    L["left-shoe"], L["right-shoe"]])
        zona[conservar] = 0

    return cv2.morphologyEx(zona, cv2.MORPH_CLOSE, np.ones((9, 9), np.uint8)), plan


GRIS = 128
def construir_agnostica(img, zona):
    a = np.asarray(img).copy()
    a[zona > 0] = GRIS
    return Image.fromarray(a)

def mascara_prenda_plana(img):
    """Silueta de la prenda de catalogo: fondo claro uniforme -> umbral + mayor componente."""
    a = np.asarray(img.convert("RGB")).astype(np.int16)
    borde = np.concatenate([a[0], a[-1], a[:, 0], a[:, -1]])
    dist = np.abs(a - np.median(borde, axis=0)).sum(2)
    m = cv2.morphologyEx((dist > 38).astype(np.uint8), cv2.MORPH_CLOSE, np.ones((7, 7), np.uint8))
    n, et, st, _ = cv2.connectedComponentsWithStats(m, 8)
    if n > 1:
        m = (et == 1 + int(np.argmax(st[1:, cv2.CC_STAT_AREA]))).astype(np.uint8)
    return cv2.morphologyEx(m, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))

print(f"{'prenda':44s} {'escote':24s} {'manga':20s} cue  man  lar")
for ruta, meta in PRENDAS_TORSO[:12]:
    pl = plan_de_borrado(meta)
    print(f"{meta['archivo'][:43]:44s} {str(meta['neck_type'])[:23]:24s} "
          f"{str(meta['sleeve_length'])[:19]:20s} {pl['cuello']:.2f} {pl['manga']:.2f} {pl['largo']:.2f}")

---
## Celda 6 · Control: ¿está corregido el bug?

**Esta es la celda que decide si seguir.** Compara, sobre una persona **vestida con ropa
normal** (no un bodysuit), la máscara vieja contra la nueva para tres prendas de escote y manga
muy distintos.

- La fila **vieja** sale idéntica en las tres columnas: no depende de la prenda, solo de la ropa
  que la persona ya lleva. Por eso las tres prendas salían con la misma silueta.
- La fila **nueva** debe cambiar en cada columna: sube al cuello en la de cuello alto, recorre el
  brazo en la de manga larga, se ciñe al torso en la de tirantes.

Si las tres columnas nuevas salen iguales, el `assert` detiene el cuaderno aquí — antes de
gastar la hora de destilación.

In [ ]:
def mascara_vieja(parse):
    """Reproduccion del area_borrar del script anterior, solo para comparar."""
    ropa = np.isin(parse, [L["upper-clothes"], L["dress"]]).astype(np.uint8)
    brazos = np.isin(parse, [L["left-arm"], L["right-arm"]]).astype(np.uint8)
    cara = np.isin(parse, [L["hat"], L["hair"], L["sunglasses"], L["face"]])
    area = cv2.dilate(ropa, np.ones((9, 9), np.uint8), 1)
    area |= (brazos & cv2.dilate(ropa, np.ones((16, 16), np.uint8), 1))
    area[cara] = 0
    return area

def elegir(frag):
    for r, m in PRENDAS_TORSO:
        if frag in m["archivo"]:
            return r, m
    raise KeyError(f"no encuentro {frag}")

# Contraste maximo, verificado contra el manifiesto: cuello 1.00/0.70/0.35, manga 1.00/0.45/0.14
MUESTRA = ["sueter_marinero_rayas_cuello_zipper",
           "camiseta_basica_ajustada_azul_marino",
           "vestido_mini_tirantes_crochet_marron_terracota"]

# Se prueba sobre personas VESTIDAS, el caso real de la clienta. Se evaluan VARIAS y no
# una sola: en alguien con vestido largo o abrigo, la ropa previa dilatada ya abre casi toda
# la zona y la adaptacion por prenda queda tapada. Eso es correcto -hay que borrarla entera
# y sobra sitio-, pero haria fallar un control hecho sobre una unica persona.
cand = [k for k in VESTIDOS[:80] if analizar_vestida(k)[2] is not None]
assert len(cand) >= 8, "Muy pocas personas vestidas con pose detectable"

def _rango_de(k):
    _, par, pos = analizar_vestida(k)
    a = [float(mascara_agnostica_por_prenda(par, pos, elegir(f)[1])[0].mean()) for f in MUESTRA]
    return max(a) - min(a)

evaluadas = [(k, _rango_de(k)) for k in cand[:20]]
adaptan = [k for k, r in evaluadas if r > 0.01]
print(f"Personas evaluadas      : {len(evaluadas)}")
print(f"La mascara se adapta en : {len(adaptan)}/{len(evaluadas)} "
      f"({100*len(adaptan)/len(evaluadas):.0f}%)")
print("Las que no, llevan vestido o abrigo: la ropa previa ya abre toda la zona.")

# Para la figura se usa una en la que SI se aprecie la diferencia.
k_demo = adaptan[0] if adaptan else cand[0]
img_v, parse_v, pose_v = analizar_vestida(k_demo)
vieja = mascara_vieja(parse_v)

fig, ax = plt.subplots(4, 3, figsize=(12, 15))
areas, cuellos = [], []
for j, frag in enumerate(MUESTRA):
    rp, mt = elegir(frag)
    nueva, plan = mascara_agnostica_por_prenda(parse_v, pose_v, mt)
    areas.append(float(nueva.mean())); cuellos.append(plan["cuello"])

    ax[0, j].imshow(analizar(rp)[0])
    ax[0, j].set_title(f"{mt['neck_type'][:20]}\n{mt['sleeve_length'][:20]}", fontsize=9)
    ax[1, j].imshow(construir_agnostica(img_v, vieja))
    ax[1, j].set_title("VIEJA - identica siempre", fontsize=9, color="#b00")
    ax[2, j].imshow(construir_agnostica(img_v, nueva))
    ax[2, j].set_title(f"NUEVA  cue={plan['cuello']:.2f} man={plan['manga']:.2f}",
                       fontsize=9, color="#070")
    dif = np.zeros((*nueva.shape, 3), np.uint8)
    dif[..., 0] = vieja * 200          # rojo  = solo la vieja
    dif[..., 1] = nueva * 200          # verde = lo que la nueva anade
    ax[3, j].imshow(dif)
    ax[3, j].set_title("verde = espacio ganado", fontsize=9)

for a in ax.ravel():
    a.axis("off")
plt.tight_layout()
plt.savefig(SALIDA / "control_mascara.png", dpi=110, bbox_inches="tight")
plt.show()

print(f"Persona de la figura: {k_demo}")
print(f"Area liberada       : {[f'{a:.3f}' for a in areas]}")
print(f"Area mascara vieja  : {vieja.mean():.3f}  (constante para las 3)")
print(f"Fraccion de cuello  : {cuellos}")

frac = len(adaptan) / len(evaluadas)
assert frac >= 0.70, (
    f"FALLO: la mascara solo se adapta en el {100*frac:.0f}% de las personas. "
    "Revisa que pose_de() devuelva landmarks.")
assert max(cuellos) - min(cuellos) > 0.5, "FALLO: el escote no se esta diferenciando."
print("OK - la mascara depende de la prenda destino. Bugs 1, 2 y 5 corregidos en origen.")

---
# Celda 7 · MODO FOTO — CatVTON

Para la foto que sube la clienta prima la calidad, no la velocidad. **CatVTON** es Stable
Diffusion inpainting especializado en try-on, preentrenado sobre ~13.000 pares reales de
VITON-HD, donde las personas van vestidas con ropa normal — justo tu caso de uso.

**Por qué esto sí respeta el diseño de la prenda.** CatVTON recibe la prenda como referencia
directa por atención cruzada, no como un vector comprimido: copia la textura, el estampado y los
botones en lugar de reinventarlos. Y regenera **todo** lo que hay dentro de la máscara, sin
ninguna pérdida que lo ate a la silueta previa. Combinado con la máscara de la celda 5, el
escote deja de ser un problema estructural.

Descarga ~5 GB la primera vez, cacheados en Drive.

In [ ]:
if EN_KAGGLE:
    os.environ.setdefault("HF_HOME", "/tmp/hf")
    DIR_CV = Path("/tmp/CatVTON")
else:
    os.environ.setdefault("HF_HOME", str(SALIDA / "hf"))
    DIR_CV = SALIDA / "CatVTON"
Path(os.environ["HF_HOME"]).mkdir(parents=True, exist_ok=True)
if not DIR_CV.exists():
    print("Clonando CatVTON...")
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/Zheng-Chong/CatVTON", str(DIR_CV)], capture_output=True)
if str(DIR_CV) not in sys.path:
    sys.path.insert(0, str(DIR_CV))

from huggingface_hub import snapshot_download
print("Descargando pesos (~5 GB la primera vez)...")
REPO_CV = snapshot_download(repo_id="zhengchong/CatVTON")

from model.pipeline import CatVTONPipeline
pipe = CatVTONPipeline(
    base_ckpt="booksforcharlie/stable-diffusion-inpainting",
    attn_ckpt=REPO_CV, attn_ckpt_version="mix",
    weight_dtype=torch.float16, device="cuda", skip_safety_check=True)
print("Modo foto listo.")


@torch.no_grad()
def probar_prenda(persona_pil, prenda_pil, meta, pasos=30, escala=2.5, semilla=42,
                  pose=None, parse=None):
    """Coloca una prenda del catalogo sobre una persona. Es la funcion del MODO FOTO.

    `pose` y `parse` se pasan cuando ya se conocen (destilacion): asi no se recalculan y la
    pose es la precalculada, no una nueva estimacion.
    """
    persona = persona_pil if persona_pil.size == (W_DIF, H_DIF) else letterbox(persona_pil)
    if parse is None:
        parse = segmentar(persona)
    if pose is None:
        pose = pose_de(persona, parse)
    zona, plan = mascara_agnostica_por_prenda(parse, pose, meta)
    mask = Image.fromarray((zona * 255).astype(np.uint8)).convert("L")
    prenda = prenda_pil if prenda_pil.size == (W_DIF, H_DIF) else letterbox(prenda_pil)
    g = torch.Generator(device="cuda").manual_seed(semilla)
    out = pipe(persona, prenda, mask, num_inference_steps=pasos, guidance_scale=escala, generator=g)
    bruto = out[0] if isinstance(out, (list, tuple)) else out
    bruto = bruto.convert("RGB").resize(persona.size, Image.LANCZOS)
    # La difusion pasa TODA la imagen por el VAE y altera un poco lo que esta fuera de la
    # zona. Se recompone: fuera de la zona el resultado es la persona, pixel a pixel.
    suave = Image.fromarray((cv2.GaussianBlur(zona.astype(np.float32), (0, 0), 1.5) * 255)
                            .clip(0, 255).astype(np.uint8))
    res = Image.composite(bruto, persona, suave)
    return dict(resultado=res, bruto=bruto, persona=persona, prenda=prenda, zona=zona,
                parse=parse, pose=pose, plan=plan)


In [ ]:
# --- Demo del caso que fallaba: persona vestida + camiseta de manga -------------------
demo_k = cand[0]
img_demo = analizar_vestida(demo_k)[0]

fig, ax = plt.subplots(2, 4, figsize=(15, 9))
ax[0, 0].imshow(img_demo); ax[0, 0].set_title("clienta (con su ropa)", fontsize=10)
ax[1, 0].axis("off")
for j, frag in enumerate(MUESTRA, start=1):
    rp, mt = elegir(frag)
    r = probar_prenda(img_demo, analizar(rp)[0], mt, pasos=30)
    ax[0, j].imshow(r["prenda"]); ax[0, j].set_title(mt["archivo"][:22], fontsize=8)
    ax[1, j].imshow(r["resultado"]); ax[1, j].set_title("resultado", fontsize=10)
for a in ax.ravel():
    a.axis("off")
plt.tight_layout()
plt.savefig(SALIDA / "demo_modo_foto.png", dpi=110, bbox_inches="tight")
plt.show()
print("Si estas tres se ven bien, el MODO FOTO ya esta terminado: no requiere entrenamiento.")

---
# Celda 8 · Destilación — fabricar la verdad de terreno

Aquí se resuelve el defecto de fondo del modelo anterior.

Sus pérdidas comparaban la salida contra su propia prenda deformada, dentro de su propia máscara
predicha: un objetivo circular que se minimiza recoloreando, que es exactamente lo que hacía. No
había verdad de terreno porque no existía ni un solo par real.

El modo foto la **fabrica**. Cada par es `(agnóstica, prenda, máscara) → resultado`, y la mezcla
está pensada a propósito:

| Fuente | Pares | Para qué |
|---|---|---|
| 9 cuerpos HD del catálogo | ~126 | calidad y fidelidad a tus modelos de tienda |
| Personas vestidas del cache | ~300 | **que funcione con clientas reales, con su ropa** |

Sin la segunda mitad el modelo solo sabría vestir maniquíes en bodysuit.

**~45 min.** Guarda cada par al terminarlo: si Colab te desconecta, vuelve a ejecutar la celda y
continúa donde iba.

In [ ]:
PASOS_DEST   = 20      # 20 pasos: buen equilibrio calidad/tiempo para destilar
N_HD         = 126     # 9 cuerpos x 14 prendas
N_VESTIDAS   = 300     # 300 personas distintas, una prenda cada una
LIMITE_MIN   = 80      # minutos de destilacion POR SESION: al llegar se detiene limpio
SEMILLAS     = [42, 7, 1234]   # si el par falla el filtro, se reintenta con otra semilla

# Prendas de catalogo con la foto RECORTADA (como la chaqueta de la captura, cortada por el
# borde derecho): el profesor no puede vestirlas bien y el par saldria roto.
def prenda_recortada(ruta):
    """Toca un borde lateral/superior/inferior Y el contenido esta descentrado: recorte."""
    im = ImageOps.exif_transpose(Image.open(ruta)).convert("RGB")
    im.thumbnail((256, 256))
    m = mascara_prenda_plana(im).astype(bool)
    if m.sum() < 200:
        return False
    h, w = m.shape
    ys, xs = np.nonzero(m)
    descentrado = abs((xs.min() + xs.max()) / 2 - w / 2) / w > 0.04
    toca_lado = (m[:, 0].sum() > 0.03 * h) or (m[:, -1].sum() > 0.03 * h)
    toca_arriba_abajo = (m[0].sum() > 0.05 * w) or (m[-1].sum() > 0.05 * w)
    return bool((toca_lado and descentrado) or toca_arriba_abajo)

RECORTADAS = sorted(m["archivo"] for r, m in PRENDAS_TORSO if prenda_recortada(r))
print(f"Prendas con foto recortada (se excluyen de la destilacion): {len(RECORTADAS)}")
for n in RECORTADAS:
    print("   ", n)
PRENDAS_OK = [(r, m) for r, m in PRENDAS_TORSO if m["archivo"] not in set(RECORTADAS)]

# Muestreo estratificado de prendas: un corte simple seguiria el orden alfabetico de
# carpeta y se quedaria sin un solo vestido (25 de las 60 de torso).
por_cat = defaultdict(list)
for r, m in PRENDAS_OK:
    por_cat[m["category"]].append((r, m))
prendas_dest = [por_cat[c][0] for c in sorted(por_cat)]          # 1 de cada categoria
sobrantes = [x for c in sorted(por_cat) for x in por_cat[c][1:]]
random.Random(42).shuffle(sobrantes)
prendas_dest += sobrantes[:max(0, 30 - len(prendas_dest))]
print(f"Prendas para destilar: {len(prendas_dest)}")
for c, n in sorted(Counter(m["category"] for _, m in prendas_dest).items()):
    print(f"   {c:30s} {n}")

# --- lista de tareas: DETERMINISTA (misma semilla), asi cada sesion continua donde quedo ---
tareas = []
for i, (rc, _) in enumerate(CUERPOS_HD):                         # cuerpos HD
    for rp, mt in prendas_dest[:N_HD // max(1, len(CUERPOS_HD))]:
        tareas.append(("hd", rc, rp, mt))

vest_ok = list(VESTIDOS)
random.Random(7).shuffle(vest_ok)
rnd = random.Random(11)
for k in vest_ok[:N_VESTIDAS * 2]:                               # x2 por si alguna no da pose
    rp, mt = rnd.choice(prendas_dest)
    tareas.append(("vest", k, rp, mt))

print(f"\nTareas totales: {len(tareas)}  (se cortara al llegar a {N_HD + N_VESTIDAS} pares)")


In [ ]:
# ===== DESTILACION CON FILTRO DE CALIDAD (reanudable) =====================================
# Regla: un par solo entra al entrenamiento si el profesor (CatVTON) lo hizo BIEN. Es lo que
# faltaba: la captura del error mostraba una chaqueta guinda de manga larga convertida en una
# camiseta corta marron semitransparente, y el alumno la aprendia tal cual.
#
# Reanudacion: cada par aceptado es un juego de archivos (el *_res.png se escribe el ULTIMO),
# y cada intento queda en filtro_registro.jsonl. Si Colab se cierra, al volver a ejecutar esta
# celda se salta todo lo aceptado Y lo ya rechazado, y sigue por el resto.
import json as _json

U = dict(fuga=12.0,        # dif. media (0-255) FUERA de la zona en la salida bruta del profesor
         color=30.0,       # distancia Lab entre la prenda del catalogo y la ropa nueva
         cobertura=0.25,   # fraccion de la zona que cubre la ropa nueva
         piel_extra=0.15,  # piel "de mas" dentro de la prenda (malla semitransparente)
         manga_larga=0.30, # con manga larga, maximo de brazo al aire
         sin_mangas=0.45)  # sin mangas, minimo de brazo al aire

def _lab_mediana(rgb, mascara):
    if int(mascara.sum()) < 30:
        return None
    lab = cv2.cvtColor(rgb.astype(np.float32) / 255.0, cv2.COLOR_RGB2LAB)
    return np.median(lab[mascara > 0], axis=0)

def _piel(rgb):
    ycc = cv2.cvtColor(rgb, cv2.COLOR_RGB2YCrCb)
    cr, cb = ycc[..., 1], ycc[..., 2]
    return (cr >= 133) & (cr <= 173) & (cb >= 77) & (cb <= 127)

def verificar_par(r, meta):
    """(aceptado, motivos, metricas). Solo mira el resultado; no usa el entrenamiento."""
    motivos, m = [], {}
    persona = np.asarray(r["persona"].convert("RGB"))
    bruto = np.asarray(r["bruto"].convert("RGB"))
    res = np.asarray(r["resultado"].convert("RGB"))
    zona = r["zona"].astype(np.uint8)

    # 1. filtracion: el profesor no debe tocar lo que esta lejos de la zona
    lejos = cv2.dilate(zona, np.ones((9, 9), np.uint8)) == 0
    m["fuga"] = float(np.abs(bruto.astype(np.int16) - persona.astype(np.int16)).mean(2)[lejos].mean())
    if m["fuga"] > U["fuga"]:
        motivos.append("fuga")

    # 2. cobertura: tiene que haber ropa nueva llenando la zona
    parse_res = segmentar(r["resultado"])
    ropa = np.isin(parse_res, [L["upper-clothes"], L["dress"]]) & (zona > 0)
    m["cobertura"] = float(ropa.sum()) / max(1, int(zona.sum()))
    if m["cobertura"] < U["cobertura"]:
        motivos.append("cobertura")

    # 3. color: guinda -> marron es un fallo
    pren = np.asarray(r["prenda"].convert("RGB"))
    mpren = mascara_prenda_plana(r["prenda"])
    lab_p, lab_r = _lab_mediana(pren, mpren), _lab_mediana(res, ropa.astype(np.uint8))
    if lab_r is None:
        motivos.append("sin_ropa")
    elif lab_p is not None:                       # prenda blanca sobre blanco: no se puede medir
        m["color"] = float(np.linalg.norm(lab_p - lab_r))
        if m["color"] > U["color"]:
            motivos.append("color")

    # 4. opacidad: piel donde deberia haber tela (malla semitransparente)
    if ropa.sum() > 50:
        piel_prenda = float(_piel(pren)[mpren > 0].mean()) if mpren.sum() > 50 else 0.0
        m["piel_extra"] = float(_piel(res)[ropa].mean()) - piel_prenda
        if m["piel_extra"] > U["piel_extra"]:
            motivos.append("transparente")

    # 5. manga: el brazo al aire debe corresponder al largo de manga de la prenda
    manga = norm(meta.get("sleeve_length"))
    pose = r["pose"]
    if pose is not None:
        pt = lambda k: pose[P[k]][:2]
        ancho = float(np.linalg.norm(pt("hombro_i") - pt("hombro_d"))) or (W_DIF * 0.25)
        caps = np.zeros(zona.shape, np.uint8)
        for lado in ("i", "d"):
            _capsula(caps, pt(f"hombro_{lado}"), pt(f"codo_{lado}"), 0.15 * ancho)
            _capsula(caps, pt(f"codo_{lado}"), pt(f"muneca_{lado}"), 0.12 * ancho)
        if caps.sum() > 200:
            al_aire = np.isin(parse_res, [L["left-arm"], L["right-arm"]]) & (caps > 0)
            m["piel_brazo"] = float(al_aire.sum()) / float(caps.sum())
            if "larga" in manga and m["piel_brazo"] > U["manga_larga"]:
                motivos.append("manga_deberia_ser_larga")
            if "sin mangas" in manga and "cap" not in manga and m["piel_brazo"] < U["sin_mangas"]:
                motivos.append("manga_deberia_ser_nula")
    return (not motivos), motivos, m


REG = SALIDA / "filtro_registro.jsonl"

def registrar(nombre, estado, semilla, motivos, metricas, tipo, categoria):
    with open(REG, "a", encoding="utf-8") as f:
        f.write(_json.dumps(dict(par=nombre, estado=estado, semilla=semilla, motivos=motivos,
                                 metricas=metricas, tipo=tipo, categoria=categoria),
                            ensure_ascii=False) + "\n")
        f.flush(); os.fsync(f.fileno())

def leer_registro():
    if not REG.exists():
        return []
    filas = []
    for linea in REG.read_text(encoding="utf-8").splitlines():
        try:
            filas.append(_json.loads(linea))
        except Exception:
            pass                                    # linea a medias por un corte: se ignora
    return filas

OBJETIVO_PARES = N_HD + N_VESTIDAS
aceptados = {p.name[:-8] for p in DIR_PARES.glob("*_res.png")}
rechazados = {f["par"] for f in leer_registro() if f["estado"] == "rechazado"}
print(f"Ya aceptados: {len(aceptados)}  ·  ya rechazados (no se reintentan): {len(rechazados)}")

t0 = time.time()
fallos = oom = 0
detenido_por_tiempo = False
for tipo, orig, rp, mt in tqdm(tareas):
    if len(aceptados) >= OBJETIVO_PARES:
        break
    if time.time() - t0 > LIMITE_MIN * 60:
        detenido_por_tiempo = True
        break
    nom = f"{tipo}_{clave(orig) if tipo == 'hd' else orig}__{clave(rp)}"
    if nom in aceptados or nom in rechazados:
        continue
    try:
        if tipo == "hd":
            persona, parse, pose = analizar(orig, ident_hd(orig))
        else:
            persona, parse, pose = analizar_vestida(orig)
        if pose is None:
            registrar(nom, "rechazado", None, ["sin_pose"], {}, tipo, mt["category"])
            rechazados.add(nom)
            continue
        prenda_pil = analizar(rp)[0]

        aceptado, r, motivos = False, None, []
        for sem in SEMILLAS:
            r = probar_prenda(persona, prenda_pil, mt, pasos=PASOS_DEST, semilla=sem,
                              pose=pose, parse=parse)
            aceptado, motivos, met = verificar_par(r, mt)
            if aceptado:
                break
        if not aceptado:
            registrar(nom, "rechazado", SEMILLAS[-1], motivos, met, tipo, mt["category"])
            rechazados.add(nom)
            continue

        # Se guarda a 192x256, la resolucion del modo camara. El *_res.png va el ULTIMO.
        peq = lambda im: im.resize((W, H), Image.LANCZOS)
        guardar_png(peq(construir_agnostica(r["persona"], r["zona"])), DIR_PARES / f"{nom}_agn.png")
        guardar_png(Image.fromarray((r["zona"] * 255).astype(np.uint8)).resize((W, H), Image.NEAREST),
                    DIR_PARES / f"{nom}_zona.png")
        guardar_png(Image.fromarray(r["parse"]).resize((W, H), Image.NEAREST),
                    DIR_PARES / f"{nom}_parse.png")
        esc = np.array([W / W_DIF, H / H_DIF, 1.0], np.float32)
        guardar_npy(r["pose"] * esc, DIR_PARES / f"{nom}_pose.npy")
        (DIR_PARES / f"{nom}_meta.json").write_text(
            _json.dumps({"prenda": mt["archivo"], "tipo": tipo,
                         "neck_type": mt.get("neck_type"),
                         "sleeve_length": mt.get("sleeve_length")}, ensure_ascii=False),
            encoding="utf-8")
        guardar_png(peq(r["resultado"]), DIR_PARES / f"{nom}_res.png")
        registrar(nom, "aceptado", sem, [], met, tipo, mt["category"])
        aceptados.add(nom)
    except torch.cuda.OutOfMemoryError:
        oom += 1
        torch.cuda.empty_cache()
        print("  sin memoria de GPU: se salta este par y se limpia la cache")
    except Exception as e:
        fallos += 1
        if fallos <= 5:
            print(f"  fallo ({mt['archivo'][:28]}): {str(e)[:70]}")
        continue

n_ok = len(list(DIR_PARES.glob("*_res.png")))
n_hd = len(list(DIR_PARES.glob("hd_*_res.png")))
print(f"\nPares aceptados : {n_ok}   (HD {n_hd} · vestidas {n_ok - n_hd})   objetivo {OBJETIVO_PARES}")
print(f"Esta sesion     : {(time.time()-t0)/60:.1f} min  ·  fallos: {fallos}  ·  sin memoria: {oom}")
if detenido_por_tiempo or n_ok < OBJETIVO_PARES:
    print("\n>>> QUEDAN PARES POR HACER. Vuelve a ejecutar ESTA celda (o la sesion entera): "
          "retoma exactamente donde se quedo, no repite nada.")


---
## Puerta A · ¿El profesor lo está haciendo bien?

Resume qué pares se aceptaron o rechazaron y **por qué**. Si se acepta menos del 60 %, el problema
no es el alumno sino el profesor o la zona de borrado: **no entrenes hasta arreglarlo.**

In [ ]:
FORZAR_CONTINUAR = False     # solo True si revisaste el informe y decides seguir igualmente

reg = leer_registro()
ult = {}
for f in reg:                                   # el ultimo estado de cada par
    ult[f["par"]] = f
n_acc = sum(1 for f in ult.values() if f["estado"] == "aceptado")
n_rec = sum(1 for f in ult.values() if f["estado"] == "rechazado")
tasa = n_acc / max(1, n_acc + n_rec)
print(f"Aceptados {n_acc}  ·  rechazados {n_rec}  ·  tasa de aceptacion {tasa:.0%}")

motivos_c = Counter(m for f in ult.values() if f["estado"] == "rechazado" for m in f["motivos"])
print("\nMotivos de rechazo:")
for m, n in motivos_c.most_common():
    print(f"   {m:28s} {n}")

por_cat_acc = defaultdict(lambda: [0, 0])
for f in ult.values():
    por_cat_acc[f["categoria"]][0 if f["estado"] == "aceptado" else 1] += 1
print("\nPor categoria (aceptados / rechazados):")
for c, (a, r_) in sorted(por_cat_acc.items()):
    print(f"   {c:30s} {a:4d} / {r_:4d}")

fugas = [f["metricas"].get("fuga") for f in reg if f["metricas"].get("fuga") is not None]
if fugas:
    print(f"\nFuga del profesor (0-255): mediana {np.median(fugas):.1f} · p90 {np.percentile(fugas, 90):.1f} "
          f"· umbral {U['fuga']}")
    print("   Si casi todo se rechaza por 'fuga', sube U['fuga'] en la celda anterior.")

informe = SALIDA / "reporte_filtro.json"
informe.write_text(json.dumps(dict(aceptados=n_acc, rechazados=n_rec, tasa=round(tasa, 3),
                                   motivos=dict(motivos_c), umbrales=U,
                                   por_categoria={c: dict(aceptados=a, rechazados=r_)
                                                  for c, (a, r_) in por_cat_acc.items()},
                                   prendas_recortadas=RECORTADAS),
                              ensure_ascii=False, indent=2), encoding="utf-8")

# Origen de las poses usadas hasta ahora en ESTA sesion
tot_pose = sum(ORIGEN_POSE.values())
if tot_pose:
    pct_resp = ORIGEN_POSE["respaldo"] / tot_pose
    print(f"\nOrigen de las poses: {dict(ORIGEN_POSE)}  (respaldo {pct_resp:.0%})")
    assert pct_resp <= 0.15 or FORZAR_CONTINUAR, (
        f"El {pct_resp:.0%} de las poses viene del estimador de respaldo (impreciso). Revisa que "
        "06_poses_precalculadas/ este en Drive y que el nombre coincida.")

assert (n_acc + n_rec) < 40 or tasa >= 0.60 or FORZAR_CONTINUAR, (
    f"Solo se acepta el {tasa:.0%} de los pares (minimo 60%). Mira los motivos de arriba: el "
    "profesor o la zona de borrado necesitan ajuste ANTES de entrenar. (FORZAR_CONTINUAR=True para omitir)")
assert n_acc >= 150 or FORZAR_CONTINUAR, (
    f"Hay {n_acc} pares aceptados; hacen falta al menos 150. Vuelve a ejecutar la celda de "
    "destilacion (retoma donde se quedo).")
n_vest = len(list(DIR_PARES.glob("vest_*_res.png")))
assert n_vest >= 80 or FORZAR_CONTINUAR, (
    f"Solo {n_vest} personas vestidas; sin ellas el modelo solo sabria vestir bodysuits.")
print("\nPuerta A superada: se puede entrenar.")

---
## Celda 9 · Dataset del modo cámara

Lo que ve el modelo ligero en cada muestra:

| tensor | canales | origen |
|---|---|---|
| `agnostica` | 3 | persona con la zona en gris |
| `zona` | 1 | máscara a regenerar (celda 5) |
| `pose_mapa` | 3 | campos de distancia a torso, brazos y cuello |
| `prenda` | 3 | imagen plana del catálogo |
| `mascara_prenda` | 1 | silueta de la prenda |
| **`objetivo`** | 3 | **lo que produjo el modo foto** ← verdad de terreno |
| `ropa_previa` | 1 | silueta de la ropa vieja, solo para **castigarla** |

La última fila importa: la silueta de la ropa anterior sigue entrando al entrenamiento, pero ha
cambiado de bando. Antes era el objetivo del Dice; ahora es lo que se penaliza.

**Aumentación fuerte** — escala, traslación, espejo, brillo y tono — porque aunque hay ~300
personas distintas, conviene cubrir más variedad de encuadres de los que dará una cámara de
móvil sujeta a pulso.

In [ ]:
from torch.utils.data import Dataset, DataLoader

def mapa_de_pose(pose, h=H, w=W):
    """3 canales: torso, brazos, cuello. Geometria corporal sin mirar la ropa."""
    m = np.zeros((3, h, w), np.float32)
    if pose is None:
        return m
    pt = lambda k: pose[P[k]][:2]
    hi, hd = pt("hombro_i"), pt("hombro_d")
    ancho = float(np.linalg.norm(hi - hd)) or (w * 0.25)
    t = np.zeros((h, w), np.uint8)
    cv2.fillPoly(t, [np.array([hi, hd, pt("cadera_d"), pt("cadera_i")], np.int32)], 1)
    m[0] = cv2.GaussianBlur(t.astype(np.float32), (0, 0), max(1.0, ancho * 0.12))
    b = np.zeros((h, w), np.uint8)
    for lado in ("i", "d"):
        for a, c in ((f"hombro_{lado}", f"codo_{lado}"), (f"codo_{lado}", f"muneca_{lado}")):
            cv2.line(b, tuple(pt(a).astype(int)), tuple(pt(c).astype(int)), 1,
                     int(max(2, ancho * 0.16)))
    m[1] = cv2.GaussianBlur(b.astype(np.float32), (0, 0), max(1.0, ancho * 0.10))
    cu = np.zeros((h, w), np.uint8)
    boca = (pt("boca_i") + pt("boca_d")) / 2.0
    cv2.line(cu, tuple(boca.astype(int)), tuple(((hi + hd) / 2).astype(int)), 1,
             int(max(2, ancho * 0.28)))
    m[2] = cv2.GaussianBlur(cu.astype(np.float32), (0, 0), max(1.0, ancho * 0.10))
    return np.clip(m, 0, 1)

def a_tensor(img):
    a = np.asarray(img.convert("RGB"), np.float32) / 127.5 - 1.0
    return torch.from_numpy(a).permute(2, 0, 1)

def a_imagen(t):
    a = (t.detach().cpu().float().permute(1, 2, 0).numpy() + 1) * 127.5
    return Image.fromarray(np.clip(a, 0, 255).astype(np.uint8))

PRENDA_POR_ARCHIVO = {m["archivo"]: r for r, m in PRENDAS_TORSO}

class DatasetDestilado(Dataset):
    def __init__(self, nombres, entrenar=True):
        self.nombres, self.entrenar = nombres, entrenar
        self.cache_prenda = {}

    def __len__(self):
        return len(self.nombres) * (6 if self.entrenar else 1)

    def _prenda(self, archivo):
        if archivo not in self.cache_prenda:
            im = letterbox(Image.open(PRENDA_POR_ARCHIVO[archivo])).resize((W, H), Image.LANCZOS)
            self.cache_prenda[archivo] = (im, mascara_prenda_plana(im))
        return self.cache_prenda[archivo]

    def __getitem__(self, i):
        nom = self.nombres[i % len(self.nombres)]
        d = DIR_PARES
        try:
            agn = Image.open(d / f"{nom}_agn.png").convert("RGB")
            obj = Image.open(d / f"{nom}_res.png").convert("RGB")
            zona = np.array(Image.open(d / f"{nom}_zona.png").convert("L")) > 127
            parse = np.array(Image.open(d / f"{nom}_parse.png"))
            pose = np.load(d / f"{nom}_pose.npy")
            meta = json.loads((d / f"{nom}_meta.json").read_text(encoding="utf-8"))
            prenda_img, mpr = self._prenda(meta["prenda"])

            if self.entrenar:
                if random.random() < 0.5:                      # espejo
                    agn, obj = ImageOps.mirror(agn), ImageOps.mirror(obj)
                    prenda_img = ImageOps.mirror(prenda_img)
                    zona, parse = zona[:, ::-1].copy(), parse[:, ::-1].copy()
                    mpr = mpr[:, ::-1].copy()
                    pose = pose.copy(); pose[:, 0] = W - pose[:, 0]

                if random.random() < 0.7:                      # escala + traslacion
                    s = 0.88 + random.random() * 0.24
                    dx = int((random.random() - 0.5) * 0.10 * W)
                    dy = int((random.random() - 0.5) * 0.10 * H)
                    M = np.float32([[s, 0, (1 - s) * W / 2 + dx], [0, s, (1 - s) * H / 2 + dy]])
                    war = lambda a, f=cv2.INTER_LINEAR, b=(GRIS, GRIS, GRIS): cv2.warpAffine(
                        a, M, (W, H), flags=f, borderMode=cv2.BORDER_CONSTANT, borderValue=b)
                    agn = Image.fromarray(war(np.asarray(agn)))
                    obj = Image.fromarray(war(np.asarray(obj)))
                    zona = war(zona.astype(np.uint8), cv2.INTER_NEAREST, 0).astype(bool)
                    parse = war(parse, cv2.INTER_NEAREST, 0)
                    pose = pose.copy()
                    pose[:, 0] = pose[:, 0] * s + (1 - s) * W / 2 + dx
                    pose[:, 1] = pose[:, 1] * s + (1 - s) * H / 2 + dy

                if random.random() < 0.6:                      # brillo y temperatura
                    f = 0.85 + random.random() * 0.30
                    tono = np.array([1.0 + (random.random() - 0.5) * 0.10, 1.0,
                                     1.0 + (random.random() - 0.5) * 0.10], np.float32)
                    ap = lambda im: Image.fromarray(np.clip(
                        np.asarray(im, np.float32) * f * tono, 0, 255).astype(np.uint8))
                    agn, obj = ap(agn), ap(obj)

            pm = mapa_de_pose(pose)
            # El escote y las mangas pesan mas: son los dos errores del modelo viejo.
            peso = np.ones((1, H, W), np.float32)
            peso += 5.0 * pm[2:3]
            brazos = np.isin(parse, [L["left-arm"], L["right-arm"]]).astype(np.float32)
            peso += 3.0 * (brazos * zona)[None]
            previa = np.isin(parse, [L["upper-clothes"], L["dress"]]).astype(np.float32)

            # Banda de costura: 4 px dentro del borde de la zona. Ahi el resultado se une con
            # la persona; un error de tono en esa banda es lo que dibujaba el "recuadro".
            z8 = zona.astype(np.uint8)
            banda = (z8 - cv2.erode(z8, np.ones((9, 9), np.uint8))).astype(np.float32)
            peso += 4.0 * banda[None]

            return dict(banda=torch.from_numpy(banda)[None],
agnostica=a_tensor(agn),
                        zona=torch.from_numpy(zona.astype(np.float32))[None],
                        pose_mapa=torch.from_numpy(pm),
                        prenda=a_tensor(prenda_img),
                        mascara_prenda=torch.from_numpy(mpr.astype(np.float32))[None],
                        objetivo=a_tensor(obj),
                        peso=torch.from_numpy(peso),
                        ropa_previa=torch.from_numpy(previa)[None])
        except Exception:
            return None

def collate(b):
    b = [x for x in b if x is not None]
    return torch.utils.data.default_collate(b) if b else None

nombres = sorted({p.name[:-8] for p in DIR_PARES.glob("*_res.png")})

# Division ESTABLE: la decide el nombre del par, no el orden ni cuantos pares haya. Si entre
# dos sesiones se destilan mas pares, ninguno cambia de bando (no hay fuga entre entreno y
# validacion) y un entrenamiento reanudado sigue siendo comparable.
def es_validacion(nombre):
    return int(hashlib.md5(nombre.encode()).hexdigest(), 16) % 10 == 0

nom_va = [n for n in nombres if es_validacion(n)]
nom_tr = [n for n in nombres if not es_validacion(n)]
assert len(nom_va) >= 4, "Muy pocos pares de validacion; destila mas."
LOAD_TR = DataLoader(DatasetDestilado(nom_tr, True), batch_size=12, shuffle=True,
                     num_workers=2, collate_fn=collate, drop_last=True, persistent_workers=True)
LOAD_VA = DataLoader(DatasetDestilado(nom_va, False), batch_size=8, shuffle=False,
                     num_workers=2, collate_fn=collate)
print(f"Pares {len(nombres)}  ->  entreno {len(nom_tr)} (x6 aumentado) · validacion {len(nom_va)}")
print(f"Lotes por epoca: {len(LOAD_TR)}")

---
# Celda 10 · MODO CÁMARA — red ligera

Objetivo: **30 fps en el móvil**. Eso descarta la difusión (8-15 s por fotograma) y también las
dos ResNet34 del modelo anterior (~44 M de parámetros). Presupuesto: **~1,9 M a 192×256**.

```
 entradas (11 canales)          ┌──────────────┐
 agnostica 3 ─┐                 │   encoder    │  4 escalas · 32/64/128/192
 zona      1 ─┤                 │  separable   │  convolución separable (MobileNet)
 pose_mapa 3 ─┼──► concat ─────►│              │
 prenda    3 ─┤                 └──────┬───────┘
 masc_pr   1 ─┘                        │
                        ┌──────────────┼──────────────┐
                        ▼              ▼              ▼
                  cabeza FLUJO    decoder U-Net   cabeza ALFA
                  ±0.60 (era      salida RGB      fusión
                  ±0.22, bug 3)
```

**Tres diferencias con el modelo roto:**

1. `tanh(flujo) * 0.60` en vez de `* 0.22`. El límite anterior hacía físicamente imposible
   estirar una prenda plana hasta un torso o alargar una manga.
2. La composición final se hace dentro de **`zona`**, no dentro de la máscara deformada. Ya no
   hay forma de que la silueta previa recorte el resultado.
3. La cabeza de alfa aprende la fusión; no hay `conservar_total` cableado a mano repintando la
   foto original encima.

Y se añade el **PatchGAN**: un discriminador de 70×70 que juzga parches locales. Sin él,
`L1 + VGG` converge al promedio de las soluciones plausibles, y el promedio de muchas texturas
es una mancha. Es la causa del desenfoque, no un adorno.

In [ ]:
class ConvSep(nn.Module):
    """Convolucion separable en profundidad: el grueso del ahorro para llegar a 30 fps."""
    def __init__(self, ent, sal, paso=1):
        super().__init__()
        self.f = nn.Sequential(
            nn.Conv2d(ent, ent, 3, paso, 1, groups=ent, bias=False),
            nn.BatchNorm2d(ent), nn.ReLU6(inplace=True),
            nn.Conv2d(ent, sal, 1, bias=False),
            nn.BatchNorm2d(sal), nn.ReLU6(inplace=True))
    def forward(self, x):
        return self.f(x)

def rejilla(h, w, dev):
    y, x = torch.meshgrid(torch.linspace(-1, 1, h, device=dev),
                          torch.linspace(-1, 1, w, device=dev), indexing="ij")
    return torch.stack([x, y], -1)[None]

def deformar(x, flujo, relleno="zeros"):
    b, _, h, w = x.shape
    base = rejilla(h, w, x.device).expand(b, -1, -1, -1)
    f = F.interpolate(flujo, size=(h, w), mode="bilinear", align_corners=False)
    return F.grid_sample(x, base + f.permute(0, 2, 3, 1), mode="bilinear",
                         padding_mode=relleno, align_corners=False)

CANALES = [32, 64, 128, 192]

class VestidorCamara(nn.Module):
    ENTRADAS = 3 + 1 + 3 + 3 + 1
    def __init__(self):
        super().__init__()
        c = CANALES
        self.tallo = nn.Sequential(nn.Conv2d(self.ENTRADAS, c[0], 3, 2, 1, bias=False),
                                   nn.BatchNorm2d(c[0]), nn.ReLU6(inplace=True))
        self.e1, self.e2, self.e3 = ConvSep(c[0], c[1], 2), ConvSep(c[1], c[2], 2), ConvSep(c[2], c[3], 2)
        self.cab_flujo = nn.Sequential(ConvSep(c[3], 64), nn.Conv2d(64, 2, 3, 1, 1))
        nn.init.zeros_(self.cab_flujo[-1].weight); nn.init.zeros_(self.cab_flujo[-1].bias)
        self.d3 = ConvSep(c[3] + c[2], c[2])
        self.d2 = ConvSep(c[2] + c[1], c[1])
        self.d1 = ConvSep(c[1] + c[0], c[0])
        self.salida = nn.Sequential(ConvSep(c[0] + 3 + 1 + 3, 48), nn.Conv2d(48, 4, 3, 1, 1))

    def forward(self, agnostica, zona, pose_mapa, prenda, mascara_prenda):
        x = torch.cat([agnostica, zona, pose_mapa, prenda, mascara_prenda], 1)
        f0 = self.tallo(x)
        f1 = self.e1(f0)
        f2 = self.e2(f1)
        f3 = self.e3(f2)

        # BUG 3 CORREGIDO: rango 0.60 (antes 0.22, incapaz de expandir una prenda plana).
        flujo = torch.tanh(self.cab_flujo(f3)) * 0.60
        ff = F.interpolate(flujo, size=agnostica.shape[-2:], mode="bilinear", align_corners=False)
        masc_def = deformar(mascara_prenda, ff, "zeros")
        prenda_def = deformar(prenda, ff, "border") * masc_def + (1 - masc_def)

        up = lambda t, ref: F.interpolate(t, ref.shape[-2:], mode="bilinear", align_corners=False)
        d = self.d3(torch.cat([up(f3, f2), f2], 1))
        d = self.d2(torch.cat([up(d, f1), f1], 1))
        d = self.d1(torch.cat([up(d, f0), f0], 1))
        d = F.interpolate(d, size=agnostica.shape[-2:], mode="bilinear", align_corners=False)

        o = self.salida(torch.cat([d, prenda_def, masc_def, agnostica], 1))
        render, alfa = torch.tanh(o[:, :3]), torch.sigmoid(o[:, 3:4])
        vestido = alfa * prenda_def + (1 - alfa) * render
        # BUGS 1 y 2 CORREGIDOS: se compone dentro de ZONA, no de la silueta de la ropa previa.
        final = zona * vestido + (1 - zona) * agnostica
        return dict(final=final, render=render, alfa=alfa, flujo=ff,
                    prenda_def=prenda_def, masc_def=masc_def)


class PatchGAN(nn.Module):
    """Discriminador 70x70. Juzga parches locales: eso es lo que fuerza la textura nitida."""
    def __init__(self, ent=3 + 1):
        super().__init__()
        def bloque(i, o, paso, norma=True):
            capas = [nn.Conv2d(i, o, 4, paso, 1, bias=not norma)]
            if norma:
                capas.append(nn.InstanceNorm2d(o))
            capas.append(nn.LeakyReLU(0.2, inplace=True))
            return capas
        self.f = nn.Sequential(*bloque(ent, 64, 2, False), *bloque(64, 128, 2),
                               *bloque(128, 256, 2), *bloque(256, 256, 1),
                               nn.Conv2d(256, 1, 4, 1, 1))
    def forward(self, img, zona):
        return self.f(torch.cat([img, zona], 1))


modelo = VestidorCamara().to(DEV)
disc = PatchGAN().to(DEV)
print(f"Generador     : {sum(p.numel() for p in modelo.parameters())/1e6:.2f} M")
print(f"Discriminador : {sum(p.numel() for p in disc.parameters())/1e6:.2f} M  (solo entrenamiento)")
print(f"El modelo anterior tenia ~44 M y corria a 192x256 con ~12 fps.")

with torch.no_grad():
    b = {k: v.to(DEV) for k, v in next(iter(LOAD_VA)).items()}
    o = modelo(b["agnostica"], b["zona"], b["pose_mapa"], b["prenda"], b["mascara_prenda"])
    print("\nSalida:", {k: tuple(v.shape) for k, v in o.items()})

---
# Celda 11 · Castigos

Cada fallo concreto del modelo anterior tiene aquí su penalización con nombre propio:

| castigo | qué error castiga | peso |
|---|---|---|
| `forma_vieja` | que la prenda generada se parezca a la silueta de la ropa que la clienta ya llevaba — **el bug principal, invertido** | 12,0 |
| `escote` | error en la banda del cuello, donde fallaba siempre | 6,0 |
| `recorte` | generar menos área de prenda que la referencia (mangas comidas) | 8,0 |
| `diseño` | que el color y el contraste de la prenda generada se aparten de la prenda real — **que no te cambie el estampado** | 4,0 |
| `bordes` | pérdida Sobel: castiga el desenfoque de contornos | 2,0 |
| `temblor` | flujo poco suave ⇒ parpadeo entre fotogramas en el modo cámara | 1,5 |
| `fuga` | pintar fuera de `zona`, sobre la cara o el fondo | 10,0 |
| `adversarial` | que el PatchGAN distinga el resultado de la referencia | 0,8 |

El primero merece una nota. Antes era `dice = 1 − Dice(máscara_generada, silueta_vieja)`, o sea
**recompensa** por parecerse a la ropa anterior. Ahora se mide cuánto se solapa la prenda
generada con la vieja **por encima** de lo que se solapa la referencia del modo foto, y ese
exceso se penaliza. Si la referencia legítimamente ocupa esa zona no hay castigo; si el modelo
se está *pegando* a la silueta vieja, sí lo hay.

El de `diseño` responde a tu requisito de que la prenda no se modifique: compara media y
desviación de color de la región generada contra la prenda real. No fuerza geometría —eso lo
marca el objetivo destilado—, solo impide que reinvente colores.

In [ ]:
from torchvision.models import vgg16, VGG16_Weights

class PerdidaVGG(nn.Module):
    def __init__(self):
        super().__init__()
        self.capas = vgg16(weights=VGG16_Weights.IMAGENET1K_V1).features[:16].eval()
        for p in self.capas.parameters():
            p.requires_grad = False
        self.register_buffer("mu", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("sd", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))
        self.hitos = {3: 1.0, 8: 0.8, 15: 0.6}
    def forward(self, x, y):
        x = ((x + 1) / 2 - self.mu) / self.sd
        y = ((y + 1) / 2 - self.mu) / self.sd
        tot = 0.0
        for i, c in enumerate(self.capas):
            x, y = c(x), c(y)
            if i in self.hitos:
                tot = tot + self.hitos[i] * F.l1_loss(x, y.detach())
        return tot

VGG = PerdidaVGG().to(DEV)

_kx = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]], device=DEV).view(1, 1, 3, 3)
_ky = _kx.transpose(2, 3).contiguous()

def sobel(x):
    """Magnitud de gradiente por canal: mide si los contornos estan definidos."""
    b, c, h, w = x.shape
    x = x.reshape(b * c, 1, h, w)
    gx = F.conv2d(x, _kx, padding=1)
    gy = F.conv2d(x, _ky, padding=1)
    return torch.sqrt(gx ** 2 + gy ** 2 + 1e-6).reshape(b, c, h, w)


def perdidas_generador(o, b, logit_falso=None):
    final, obj, zona = o["final"], b["objetivo"], b["zona"]
    peso, previa = b["peso"], b["ropa_previa"]
    area_z = zona.sum(dim=[1, 2, 3]) + 1e-5

    l1 = (((final - obj).abs() * peso * zona).sum(dim=[1, 2, 3]) /
          (peso * zona).sum(dim=[1, 2, 3]).clamp_min(1e-5)).mean()
    vgg = VGG(final * zona, obj * zona)

    # 1. parecerse a la silueta de la ropa anterior
    gen = (final - b["agnostica"]).abs().mean(1, keepdim=True).clamp(0, 1) * zona
    ref = (obj - b["agnostica"]).abs().mean(1, keepdim=True).clamp(0, 1) * zona
    s_gen = (gen * previa).sum(dim=[1, 2, 3]) / area_z
    s_ref = (ref * previa).sum(dim=[1, 2, 3]) / area_z
    c_forma = 12.0 * F.relu(s_gen - s_ref - 0.02).mean()

    # 2. escote
    cuello = b["pose_mapa"][:, 2:3] * zona
    c_escote = 6.0 * (((final - obj).abs() * cuello).sum(dim=[1, 2, 3]) /
                      cuello.sum(dim=[1, 2, 3]).clamp_min(1e-5)).mean()

    # 3. recortar la prenda
    c_recorte = 8.0 * F.relu(
        (ref.sum(dim=[1, 2, 3]) - gen.sum(dim=[1, 2, 3])) / area_z - 0.03).mean()

    # 4. preservar el diseno: estadisticas de color contra la prenda REAL
    mt = b["mascara_prenda"]
    n_p = mt.sum(dim=[2, 3]).clamp_min(1.0)
    med_p = (b["prenda"] * mt).sum(dim=[2, 3]) / n_p
    var_p = ((b["prenda"] - med_p[:, :, None, None]) ** 2 * mt).sum(dim=[2, 3]) / n_p
    mg = (o["masc_def"] * zona)
    n_g = mg.sum(dim=[2, 3]).clamp_min(1.0)
    med_g = (final * mg).sum(dim=[2, 3]) / n_g
    var_g = ((final - med_g[:, :, None, None]) ** 2 * mg).sum(dim=[2, 3]) / n_g
    c_diseno = 4.0 * ((med_g - med_p).abs().mean() +
                      (var_g.sqrt() - var_p.sqrt()).abs().mean())

    # 5. bordes definidos (anti-desenfoque)
    c_bordes = 2.0 * (((sobel(final) - sobel(obj)).abs() * zona).sum(dim=[1, 2, 3]) /
                      (zona.sum(dim=[1, 2, 3]) * 3).clamp_min(1e-5)).mean()

    # 5b. costura: en la banda de union con la persona, el tono y el contorno deben coincidir
    #     con el objetivo (si no, se ve el recuadro)
    banda = b["banda"]
    nb_ = banda.sum(dim=[1, 2, 3]).clamp_min(1e-5)
    c_costura = 4.0 * ((((final - obj).abs() * banda).sum(dim=[1, 2, 3]) / (nb_ * 3)).mean() +
                       (((sobel(final) - sobel(obj)).abs() * banda).sum(dim=[1, 2, 3]) / (nb_ * 3)).mean())

    # 6. temblor entre fotogramas
    f = o["flujo"]
    c_temblor = 1.5 * ((f[:, :, 1:] - f[:, :, :-1]).abs().mean() +
                       (f[:, :, :, 1:] - f[:, :, :, :-1]).abs().mean())

    # 7. pintar fuera de la zona
    fuera = 1.0 - zona
    c_fuga = 10.0 * (((final - b["agnostica"]).abs() * fuera).sum(dim=[1, 2, 3]) /
                     fuera.sum(dim=[1, 2, 3]).clamp_min(1e-5)).mean()

    # 8. adversarial
    c_adv = 0.8 * F.mse_loss(logit_falso, torch.ones_like(logit_falso)) \
        if logit_falso is not None else torch.zeros((), device=final.device)

    total = (3.0 * l1 + 2.0 * vgg + c_forma + c_escote + c_recorte +
             c_diseno + c_bordes + c_costura + c_temblor + c_fuga + c_adv)
    return total, dict(total=float(total), l1=float(l1), vgg=float(vgg),
                       forma_vieja=float(c_forma), escote=float(c_escote),
                       recorte=float(c_recorte), diseno=float(c_diseno),
                       bordes=float(c_bordes), costura=float(c_costura), temblor=float(c_temblor),
                       fuga=float(c_fuga), adv=float(c_adv))

print("Castigos definidos. En las curvas, 'forma_vieja' y 'recorte' deben tender a 0.")

---
## Celda 12 · Entrenamiento

~22 min. El discriminador arranca en la época 4 (`EP_GAN`): con el generador aún desorientado,
el adversarial solo mete ruido. Cada checkpoint se guarda verificando el tamaño — el `last.pth`
de 0 bytes del entrenamiento anterior nació de un guardado que falló en silencio.

In [ ]:
# ===== ENTRENAMIENTO REANUDABLE =========================================================
# Se guarda TODO el estado (modelo, discriminador, optimizadores, calendario, escaladores,
# historial, semillas) al final de CADA epoca, alternando dos archivos (ultimo_a / ultimo_b)
# de forma atomica: aunque Colab se corte MIENTRAS se escribe uno, el otro sigue valido.
# Al volver a ejecutar esta celda continua en la epoca siguiente; no hay que empezar de nuevo.
EPOCAS, EP_GAN = 40, 4

opt_g = torch.optim.AdamW(modelo.parameters(), lr=2.5e-4, weight_decay=1e-4, betas=(0.5, 0.999))
opt_d = torch.optim.AdamW(disc.parameters(), lr=1e-4, weight_decay=1e-4, betas=(0.5, 0.999))
sched = torch.optim.lr_scheduler.OneCycleLR(opt_g, max_lr=3e-4,
                                            total_steps=EPOCAS * max(1, len(LOAD_TR)),
                                            pct_start=0.20)
esc_g = torch.cuda.amp.GradScaler()
esc_d = torch.cuda.amp.GradScaler()
hist, mejor = defaultdict(list), float("inf")

def guardar(ruta, estado):
    tmp = Path(str(ruta) + ".tmp")
    torch.save(estado, tmp)
    assert tmp.stat().st_size > 1_000_000, f"checkpoint vacio ({tmp.stat().st_size} bytes)"
    tmp.replace(ruta)
    return ruta.stat().st_size

ULTIMOS = [DIR_CKPT / "ultimo_a.pth", DIR_CKPT / "ultimo_b.pth"]

def guardar_estado(ep):
    estado = dict(modelo=modelo.state_dict(), disc=disc.state_dict(),
                  opt_g=opt_g.state_dict(), opt_d=opt_d.state_dict(), sched=sched.state_dict(),
                  esc_g=esc_g.state_dict(), esc_d=esc_d.state_dict(),
                  epoca=ep, epocas=EPOCAS, n_lotes=len(LOAD_TR), hist=dict(hist), mejor=mejor,
                  rng=dict(py=random.getstate(), np=np.random.get_state(),
                           torch=torch.get_rng_state()),
                  W=W, H=H, canales=CANALES)
    guardar(ULTIMOS[ep % 2], estado)

def cargar_ultimo():
    """El estado mas reciente que se pueda leer entero, o None."""
    validos = []
    for ruta in ULTIMOS:
        if ruta.exists():
            try:
                validos.append(torch.load(ruta, map_location="cpu", weights_only=False))
            except Exception as e:
                print(f"  {ruta.name} ilegible ({str(e)[:50]}); se usa el otro.")
    return max(validos, key=lambda k: k["epoca"]) if validos else None

ini = 1
ck = cargar_ultimo()
if ck is not None:
    modelo.load_state_dict(ck["modelo"]); disc.load_state_dict(ck["disc"])
    if ck["n_lotes"] == len(LOAD_TR) and ck["epocas"] == EPOCAS:
        opt_g.load_state_dict(ck["opt_g"]); opt_d.load_state_dict(ck["opt_d"])
        sched.load_state_dict(ck["sched"])
        esc_g.load_state_dict(ck["esc_g"]); esc_d.load_state_dict(ck["esc_d"])
        hist = defaultdict(list, ck["hist"]); mejor = ck["mejor"]
        random.setstate(ck["rng"]["py"]); np.random.set_state(ck["rng"]["np"])
        torch.set_rng_state(ck["rng"]["torch"].cpu())
        ini = ck["epoca"] + 1
        print(f"REANUDO desde la epoca {ini} de {EPOCAS} (mejor validacion hasta ahora {mejor:.4f}).")
    else:
        print("El numero de pares o de epocas cambio desde el ultimo guardado: se conservan los "
              "PESOS pero el calendario de aprendizaje empieza de nuevo.")
    del ck
else:
    print("Primera vez: empiezo desde cero.")

demo = None
with torch.no_grad():                                   # panel fijo para las imagenes de control
    for lote in LOAD_VA:
        if lote is not None:
            demo = {k: v.to(DEV) for k, v in lote.items()}
            break

t0_ent = time.time()
for ep in range(ini, EPOCAS + 1):
    modelo.train(); disc.train()
    acc, n = defaultdict(float), 0
    usa_gan = ep >= EP_GAN

    for lote in LOAD_TR:
        if lote is None:
            continue
        try:
            b = {k: v.to(DEV, non_blocking=True) for k, v in lote.items()}
            with torch.autocast("cuda", torch.float16):
                o = modelo(b["agnostica"], b["zona"], b["pose_mapa"], b["prenda"], b["mascara_prenda"])
        except torch.cuda.OutOfMemoryError:
            print("  sin memoria de GPU: se salta este lote"); torch.cuda.empty_cache(); continue

        if usa_gan:                                  # --- discriminador ---
            opt_d.zero_grad(set_to_none=True)
            with torch.autocast("cuda", torch.float16):
                lr_ = disc(b["objetivo"], b["zona"])
                lf_ = disc(o["final"].detach(), b["zona"])
                pd_ = 0.5 * (F.mse_loss(lr_, torch.ones_like(lr_)) +
                             F.mse_loss(lf_, torch.zeros_like(lf_)))
            esc_d.scale(pd_).backward()
            esc_d.unscale_(opt_d)
            torch.nn.utils.clip_grad_norm_(disc.parameters(), 3.0)
            esc_d.step(opt_d); esc_d.update()

        opt_g.zero_grad(set_to_none=True)            # --- generador ---
        with torch.autocast("cuda", torch.float16):
            logit = disc(o["final"], b["zona"]) if usa_gan else None
            loss, met = perdidas_generador(o, b, logit)
        esc_g.scale(loss).backward()
        esc_g.unscale_(opt_g)
        torch.nn.utils.clip_grad_norm_(modelo.parameters(), 3.0)
        esc_g.step(opt_g); esc_g.update(); sched.step()

        bs = b["agnostica"].shape[0]
        for k, v in met.items():
            acc[k] += v * bs
        n += bs
    tr = {k: v / max(1, n) for k, v in acc.items()}

    modelo.eval()
    acc, n = defaultdict(float), 0
    with torch.no_grad():
        for lote in LOAD_VA:
            if lote is None:
                continue
            b = {k: v.to(DEV) for k, v in lote.items()}
            with torch.autocast("cuda", torch.float16):
                o = modelo(b["agnostica"], b["zona"], b["pose_mapa"], b["prenda"], b["mascara_prenda"])
                _, met = perdidas_generador(o, b, None)
            bs = b["agnostica"].shape[0]
            for k, v in met.items():
                acc[k] += v * bs
            n += bs
    va = {k: v / max(1, n) for k, v in acc.items()}

    for k in tr: hist[f"tr_{k}"].append(tr[k])
    for k in va: hist[f"va_{k}"].append(va[k])

    marca = "GAN" if usa_gan else "   "
    print(f"ep {ep:2d}/{EPOCAS} {marca}  tr {tr['total']:6.3f}  va {va['total']:6.3f}  |  "
          f"l1 {va['l1']:.3f}  forma_vieja {va['forma_vieja']:.4f}  escote {va['escote']:.3f}  "
          f"recorte {va['recorte']:.4f}  bordes {va['bordes']:.3f}  costura {va['costura']:.3f}")

    if va["total"] < mejor:
        mejor = va["total"]
        mb = guardar(DIR_CKPT / "mejor.pth",
                     dict(modelo=modelo.state_dict(), epoca=ep, loss=mejor,
                          W=W, H=H, canales=CANALES)) / 1e6
        print(f"          nuevo mejor -> mejor.pth ({mb:.1f} MB)")

    guardar_estado(ep)                               # <- lo que permite reanudar
    torch.cuda.empty_cache()

    if ep % 10 == 0 or ep == EPOCAS:
        with torch.no_grad():
            o = modelo(demo["agnostica"], demo["zona"], demo["pose_mapa"],
                       demo["prenda"], demo["mascara_prenda"])
        k = min(3, demo["agnostica"].shape[0])
        fig, ax = plt.subplots(k, 4, figsize=(12, 4 * k))
        ax = ax.reshape(k, 4)
        for i in range(k):
            for j, (t, ti) in enumerate([(demo["agnostica"][i], "agnostica"),
                                         (demo["prenda"][i], "prenda"),
                                         (o["final"][i], "modo camara"),
                                         (demo["objetivo"][i], "objetivo (modo foto)")]):
                ax[i, j].imshow(a_imagen(t)); ax[i, j].set_title(ti, fontsize=9); ax[i, j].axis("off")
        plt.tight_layout()
        plt.savefig(SALIDA / f"visual_epoca_{ep:03d}.png", dpi=95, bbox_inches="tight")
        plt.show()

print(f"\nEntrenamiento (esta sesion): {(time.time()-t0_ent)/60:.1f} min   ·   mejor validacion {mejor:.4f}")
(DIR_CKPT / "COMPLETO.json").write_text(json.dumps(dict(epocas=EPOCAS, mejor=mejor,
    fecha=time.strftime("%Y-%m-%d %H:%M"))), encoding="utf-8")

fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
ax[0].plot(hist["tr_total"], label="entreno"); ax[0].plot(hist["va_total"], label="validacion")
ax[0].set_title("perdida total"); ax[0].legend(); ax[0].grid(alpha=.3)
for k in ("va_l1", "va_vgg", "va_escote", "va_bordes"):
    ax[1].plot(hist[k], label=k.replace("va_", ""))
ax[1].set_title("reconstruccion, escote y nitidez"); ax[1].legend(); ax[1].grid(alpha=.3)
for k in ("va_forma_vieja", "va_recorte", "va_fuga", "va_diseno"):
    ax[2].plot(hist[k], label=k.replace("va_", ""))
ax[2].set_title("castigos - deben tender a 0"); ax[2].legend(); ax[2].grid(alpha=.3)
plt.tight_layout(); plt.savefig(SALIDA / "curvas.png", dpi=110); plt.show()

print(f"forma_vieja final : {hist['va_forma_vieja'][-1]:.4f}   (si sigue alto, sube su peso a 20)")
print(f"recorte final     : {hist['va_recorte'][-1]:.4f}")

---
## Puerta B · Control de calidad del alumno

Mide lo que la captura del error dejó mal y guarda `panel_control.png` para **revisarlo a ojo**.
El ONNX solo se exporta si pasan las pruebas automáticas; la revisión visual sigue siendo tuya
(mira especialmente una chaqueta de manga larga).

In [ ]:
# Se carga el MEJOR checkpoint (no el ultimo): es el que se exportara.
ck_m = torch.load(DIR_CKPT / "mejor.pth", map_location=DEV, weights_only=False)
modelo.load_state_dict(ck_m["modelo"]); modelo.eval()

m_fuera, m_delta, m_costura, n_v = [], [], [], 0
with torch.no_grad():
    for lote in LOAD_VA:
        if lote is None:
            continue
        b = {k: v.to(DEV) for k, v in lote.items()}
        o = modelo(b["agnostica"], b["zona"], b["pose_mapa"], b["prenda"], b["mascara_prenda"])
        fin, obj, zon, agn, ban = o["final"], b["objetivo"], b["zona"], b["agnostica"], b["banda"]
        # 1. fuera de la zona el resultado ES la persona
        m_fuera.append(float(((fin - agn).abs() * (1 - zon)).sum() / (1 - zon).sum().clamp_min(1)))
        # 2. tono de la prenda frente a la prenda real
        mpr = b["mascara_prenda"]; mg = (o["masc_def"] * zon)
        med_p = (b["prenda"] * mpr).sum(dim=[2, 3]) / mpr.sum(dim=[2, 3]).clamp_min(1)
        med_g = (fin * mg).sum(dim=[2, 3]) / mg.sum(dim=[2, 3]).clamp_min(1)
        m_delta.append(float((med_g - med_p).abs().mean()))
        # 3. costura: el gradiente en la banda no debe superar al del objetivo
        gs = (sobel(fin) * ban).sum() / (ban.sum() * 3).clamp_min(1)
        go = (sobel(obj) * ban).sum() / (ban.sum() * 3).clamp_min(1)
        m_costura.append(float(gs / go.clamp_min(1e-4)))
        n_v += 1

fuera_m, delta_m, cost_m = float(np.mean(m_fuera)), float(np.mean(m_delta)), float(np.mean(m_costura))
print(f"Fuera de la zona (debe ser 0)        : {fuera_m:.5f}")
print(f"Diferencia de tono con la prenda     : {delta_m:.3f}   (en [-1,1]; mejor < 0.25)")
print(f"Costura / objetivo (debe ser <= 1.5) : {cost_m:.2f}")

PUERTA_B = fuera_m < 1e-3 and delta_m < 0.25 and cost_m <= 1.5
print("\nPUERTA B (automatica):", "APROBADA" if PUERTA_B else "NO APROBADA")

# Panel de control fijo: los primeros 12 pares de validacion, siempre los mismos
ds_panel = DatasetDestilado(nom_va[:12], entrenar=False)
fig, ax = plt.subplots(len(ds_panel) // 2 + len(ds_panel) % 2, 8, figsize=(24, 3.2 * (len(ds_panel) // 2 + 1)))
ax = np.atleast_2d(ax)
for a in ax.ravel():
    a.axis("off")
with torch.no_grad():
    for n in range(len(ds_panel)):
        s = ds_panel[n]
        if s is None:
            continue
        x = {k: v[None].to(DEV) for k, v in s.items()}
        o = modelo(x["agnostica"], x["zona"], x["pose_mapa"], x["prenda"], x["mascara_prenda"])
        fila, col = n // 2, (n % 2) * 4
        for j, (t, ti) in enumerate([(s["agnostica"], "agnostica"), (s["prenda"], "prenda"),
                                     (o["final"][0].cpu(), "alumno"), (s["objetivo"], "profesor")]):
            ax[fila, col + j].imshow(a_imagen(t)); ax[fila, col + j].set_title(ti, fontsize=8)
plt.tight_layout(); plt.savefig(SALIDA / "panel_control.png", dpi=90, bbox_inches="tight"); plt.show()
(SALIDA / "puerta_b.json").write_text(json.dumps(dict(fuera=fuera_m, tono=delta_m, costura=cost_m,
    aprobada=bool(PUERTA_B)), indent=2), encoding="utf-8")
print("REVISA panel_control.png: si una chaqueta de manga larga sale corta, NO despliegues el modelo.")

---
## Celda 13 · Export ONNX verificado

El export anterior produjo un `.onnx` de 3 KB con 184 KB de pesos, cuando debía rondar los
150 MB: el grafo salió **sin pesos**, y lo que corría en producción no era el modelo entrenado.
Esta celda no confía en que el export haya ido bien — lo comprueba con tres `assert`:
tamaño en disco, equivalencia numérica con PyTorch, y fps reales.

In [ ]:
import onnx, onnxruntime as ort
assert PUERTA_B or globals().get("FORZAR_CONTINUAR", False), (
    "La puerta B no se aprobo (ver la celda anterior). No se exporta un modelo que todavia "
    "dibuja recuadros o cambia el color de la prenda.")

RUTA = DIR_ONNX / "vestidor_camara.onnx"
ck = torch.load(DIR_CKPT / "mejor.pth", map_location="cpu")
m_exp = VestidorCamara(); m_exp.load_state_dict(ck["modelo"]); m_exp.eval()

class SoloFinal(nn.Module):
    def __init__(self, m):
        super().__init__(); self.m = m
    def forward(self, agnostica, zona, pose_mapa, prenda, mascara_prenda):
        return self.m(agnostica, zona, pose_mapa, prenda, mascara_prenda)["final"]

env = SoloFinal(m_exp).eval()
ej = (torch.randn(1, 3, H, W), torch.rand(1, 1, H, W), torch.rand(1, 3, H, W),
      torch.randn(1, 3, H, W), torch.rand(1, 1, H, W))
nombres_ent = ["agnostica", "zona", "pose_mapa", "prenda", "mascara_prenda"]

torch.onnx.export(env, ej, str(RUTA), input_names=nombres_ent, output_names=["final"],
                  opset_version=17, do_constant_folding=True,
                  dynamic_axes={k: {0: "lote"} for k in nombres_ent + ["final"]})

# 1. tamano
disco = sum(f.stat().st_size for f in DIR_ONNX.glob("vestidor_camara.onnx*"))
n_par = sum(p.numel() for p in m_exp.parameters())
esperado = n_par * 4
print(f"Parametros : {n_par:,}   ->  esperado ~{esperado/1e6:.1f} MB")
print(f"En disco   : {disco/1e6:.1f} MB")
assert disco > esperado * 0.75, (
    f"EXPORT VACIO: {disco/1e6:.2f} MB frente a {esperado/1e6:.1f} esperados. "
    "Es el mismo fallo del modelo anterior; no despliegues este archivo.")
onnx.checker.check_model(onnx.load(str(RUTA)))
print("1/3 tamano y estructura OK")

# 2. equivalencia
prov = ["CUDAExecutionProvider", "CPUExecutionProvider"]
ses = ort.InferenceSession(str(RUTA), providers=prov)
ent_np = {n: t.numpy() for n, t in zip([e.name for e in ses.get_inputs()], ej)}
with torch.no_grad():
    ref = env(*ej).numpy()
err = float(np.abs(ses.run(None, ent_np)[0] - ref).max())
print(f"2/3 error maximo PyTorch vs ONNX: {err:.5f}")
assert err < 0.02, "El ONNX no reproduce al modelo; revisa el opset."

# 3. velocidad
for _ in range(10):
    ses.run(None, ent_np)
t = time.time()
for _ in range(60):
    ses.run(None, ent_np)
fps = 60 / (time.time() - t)
print(f"3/3 velocidad: {fps:.1f} fps @ {W}x{H}")
if fps < 25:
    print("    AVISO: por debajo de 25 fps en esta GPU; en movil sera bastante menos.")

(DIR_ONNX / "metadatos.json").write_text(json.dumps({
    "modelo": "vestidor_camara.onnx",
    "version": time.strftime("%Y%m%d-%H%M"),
    "arquitectura": "encoder separable + cabeza de flujo + fusion alfa, con PatchGAN",
    "parametros": int(n_par),
    "entrenado_por": "destilacion de CatVTON sobre catalogo FashionStore + personas vestidas",
    "pares": len(list(DIR_PARES.glob("*_res.png"))),
    "alto": H, "ancho": W,
    "entradas": {
        "agnostica": f"float32 [b,3,{H},{W}] en [-1,1], zona en gris 128",
        "zona": f"float32 [b,1,{H},{W}] en {{0,1}}",
        "pose_mapa": f"float32 [b,3,{H},{W}] en [0,1]: torso, brazos, cuello",
        "prenda": f"float32 [b,3,{H},{W}] en [-1,1] sobre blanco",
        "mascara_prenda": f"float32 [b,1,{H},{W}] en {{0,1}}"},
    "salidas": {"final": f"float32 [b,3,{H},{W}] en [-1,1]"},
    "fps_medido": round(fps, 1),
    "verificado": {"bytes": disco, "error_max_vs_torch": err},
}, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"\nListo: {RUTA}")
print(f"Checkpoint PyTorch: {DIR_CKPT / 'mejor.pth'}")

---
# Resumen

## Qué quedó entrenado

| Modo | Modelo | Dónde |
|---|---|---|
| **Foto** | CatVTON preentrenado, sin modificar | se descarga en la celda 7 |
| **Cámara** | Red propia ~1,9 M par., destilada | `05_entrenamiento_v3/onnx/vestidor_camara.onnx` |

Para la defensa, la respuesta honesta a *"¿qué entrenaste tú?"* es: el modelo del modo cámara,
destilado del modo foto sobre el catálogo FashionStore y sobre personas vestidas reales. CatVTON
se usa tal cual, como se usa una ResNet preentrenada.

## Los siete defectos corregidos

| # | Defecto anterior | Corrección | Celda |
|---|---|---|---|
| 1 | `dice(máscara, silueta_ropa_vieja)` premiaba copiar la forma previa | zona desde pose + metadatos de la prenda; el parecido con la ropa vieja se **castiga** | 5, 11 |
| 2 | La agnóstica solo borraba la ropa vieja dilatada | zona = superconjunto geométrico; la ropa vieja solo garantiza el borrado | 5 |
| 3 | `tanh(flujo)*0.22` | `tanh(flujo)*0.60` | 10 |
| 4 | Objetivo circular, sin verdad de terreno | **~426 pares destilados** | 8 |
| 5 | El cuello era zona preservada ⇒ escote imposible | `neck_type` decide cuánto se libera + `castigo_escote` ×6 | 5, 11 |
| 6 | `last.pth` de 0 bytes | `guardar()` verifica el tamaño | 12 |
| 7 | ONNX exportado sin pesos | tres `assert`: tamaño, equivalencia, fps | 13 |

Y dos añadidos que no venían del modelo anterior sino de tus requisitos:

- **La clienta se fotografía con su ropa** → 300 de los pares son personas reales vestidas.
- **Que no se modifique el diseño de la prenda** → `castigo_diseño` sobre las estadísticas de
  color, más CatVTON que copia la textura por atención en vez de comprimirla.

## Límites que conviene que sepas

- El modo cámara sale de **~426 pares**. Es suficiente para una demo sólida y para la defensa,
  no para producción a gran escala. Si más adelante quieres subir la calidad, lo que más rinde
  es destilar más pares (la celda 8 es reanudable: subir `N_VESTIDAS` y volver a ejecutarla).
- El modo cámara trabaja a **192×256**. Es lo que permite los 30 fps; para el resultado que la
  clienta guarda o comparte, usa siempre el modo foto.
- Las personas vestidas vienen del dataset del entrenamiento anterior, a 192×256. Eso acota la
  nitidez máxima que el modo cámara puede aprender, y es coherente con su resolución de trabajo.

## Siguiente paso

Los dos archivos que necesita el backend:

```
05_entrenamiento_v3/onnx/vestidor_camara.onnx     modo cámara
05_entrenamiento_v3/onnx/metadatos.json           contrato de entradas/salidas
```

Para el modo foto el backend llama a CatVTON con la misma `mascara_agnostica_por_prenda()` de la
celda 5 — cópiala tal cual al servicio, porque es la pieza de la que depende todo lo demás.

In [ ]:
# ===== DESCARGA DE MODELOS (KAGGLE / COLAB) ===========================
import shutil
from pathlib import Path

zip_onnx = Path("/kaggle/working/vestidor_camara_onnx.zip") if EN_KAGGLE else (SALIDA / "vestidor_camara_onnx.zip")
if DIR_ONNX.is_dir() and (DIR_ONNX / "vestidor_camara.onnx").exists():
    shutil.make_archive(str(zip_onnx).replace(".zip", ""), "zip", str(DIR_ONNX))
    print(f"✅ Modelo ONNX empaquetado: {zip_onnx} ({zip_onnx.stat().st_size / (1024*1024):.2f} MB)")
    if EN_KAGGLE:
        try:
            from IPython.display import FileLink, display
            display(FileLink(str(zip_onnx)))
        except Exception:
            pass
        print("👉 En Kaggle: También puedes descargarlo desde el panel lateral derecho: 'Output' -> /kaggle/working")
else:
    print("Nota: vestidor_camara.onnx aún no ha sido exportado. Ejecuta las celdas de entrenamiento primero.")
